# DermaReal — Experiment 3: Synthetic + Real Facial Skin Classification

This is the corrected, self-contained six-class Experiment 3 notebook. It preserves the original **prompt-wise grouping** for diffusion images (class + prompt ID, five variations per complete prompt), adds visual duplicate grouping, and runs the exact-content **MD5/SHA-256 audit on the full eligible real and synthetic pools before sampling any training mixture**.

It trains EfficientNetV2-B0, ResNet-50, DenseNet-121 and MobileNetV3-Large with the same fixed held-out validation/test sets. Each model and the validation-weighted ensemble produces real-test-only and synthetic-test-only metrics, per-class recall, prediction CSVs, checkpoints, training histories, figures and a versioned evidence ZIP.

**Three configurations:** `100_100`, `70_30`, `30_70` are fractions of each source-specific training pool, *not exact overall image ratios*. Actual source proportions are logged.

**Running on different Kaggle accounts:** The FIRST run can create splits from the images with `SPLIT_MODE='create'`. Download its `splits/splits_v3.csv` and `splits/synthetic_frozen_v1.csv`. Attach BOTH to subsequent runs and use `SPLIT_MODE='reuse'`. This retains the original splitting algorithms while guaranteeing identical assignments. The computed real split is not automatically the same as a previously completed E2 run; compare the E2 test manifest before claiming a matched E2-vs-E3 comparison.

No medical label accuracy or licensing compliance can be established from folder names alone. Review these separately before publication.

## 0 · Dependencies and environment

Installs only missing scientific packages; imports and records software versions and GPU visibility. Kaggle Internet or cached ImageNet weights may be required. All source-code cells are plain Python and can be edited in Kaggle.

In [ ]:
import sys, subprocess, importlib.util
for package, module in [('scikit-learn','sklearn'),('Pillow','PIL'),('scipy','scipy'),('matplotlib','matplotlib'),('seaborn','seaborn'),('nbformat','nbformat')]:
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable,'-m','pip','install','-q',package])

In [ ]:
import os, gc, re, json, math, time, random, shutil, hashlib, platform, zipfile, logging, sys
from pathlib import Path
from collections import Counter, defaultdict
from PIL import Image, ImageOps
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.fftpack import dct
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.metrics import (accuracy_score,balanced_accuracy_score,precision_recall_fscore_support,confusion_matrix,
    f1_score,roc_auc_score,roc_curve,cohen_kappa_score,matthews_corrcoef,classification_report)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetV2B0,ResNet50,DenseNet121,MobileNetV3Large
SEED=42
random.seed(SEED);np.random.seed(SEED);tf.random.set_seed(SEED)
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('Deterministic TF kernels unavailable:',e)
print('Python',sys.version.split()[0], '| TF',tf.__version__,'| GPUs',tf.config.list_physical_devices('GPU'))

## 1. Inputs, class-name mapping and run configuration

Attach the corrected real dataset and original synthetic dataset. The folder resolver accepts spaces and underscores. For the first run use `SPLIT_MODE='create'`. To **reuse the exact same evaluation images across other Kaggle accounts**, attach both frozen CSVs and change `SPLIT_MODE='reuse'`, `REAL_FROZEN_SPLIT_HINT`, `E3_FROZEN_SPLIT_HINT` and `MIXES_TO_RUN`. Each run is logged in `config/` and `logs/`. This configuration uses `30_70` for the first run.

In [ ]:
REAL_HINT=Path(os.environ.get('REAL_DATA_DIR', '/kaggle/input'))
SYNTH_HINT=Path(os.environ.get('SYNTHETIC_DATA_DIR', '/kaggle/input'))
# No external E2 CSV input is required; real group-aware split is reconstructed from images.
E2_ARTIFACT_HINT=None
PRIOR_E3_OUTPUT=None  # optional previous E3 output root with matching splits and past mixture outputs
E3_FROZEN_SPLIT_HINT=None  # existing canonical synthetic_frozen_v1.csv or E3 evidence ZIP; reuse for other accounts
REAL_FROZEN_SPLIT_HINT=None  # existing canonical splits_v3.csv from the same real-image dataset
# First independent session: SPLIT_MODE='create'. Other sessions: 'reuse', attach BOTH saved splits.
SPLIT_MODE='create'  # 'create' or 'reuse'; reuse requires both frozen CSVs
assert SPLIT_MODE in ('create','reuse')
if SPLIT_MODE=='reuse':
    assert REAL_FROZEN_SPLIT_HINT is not None and E3_FROZEN_SPLIT_HINT is not None, (
        'For separate Kaggle accounts attach the SAME canonical real and synthetic split CSVs.')
ROOT=Path('/kaggle/working/dermareal_e3_run001') if Path('/kaggle/working').exists() else Path('dermareal_e3_run001')
MIXES_TO_RUN=['30_70'] # choices: 100_100, 70_30, 30_70; use one per Kaggle account
RUN_TRAINING=True
RUN_SOURCE_CONTROLS=True
RUN_EFFICIENTNET_AUDIT=True
RUN_SIMILARITY_GROUPING=True # ResNet50 + 4-way pHash among synthetic candidates
BATCH_SIZE,IMG_SIZE=24,224
EPOCHS=(15,25,30)
LEARNING_RATES=(1e-3,1e-4,2e-5)
PHASH_DIST=5; EMBED_LINK_TAU=0.97; AUDIT_THRESHOLDS=(0.90,0.95)
CHUNK=160; SPLIT_TRIES=150
FOCAL_ALPHA,FOCAL_GAMMA=0.25,2.0
L2_REG,DROPOUT=1e-4,0.4
VALID_EXTS={'.jpg','.jpeg','.png','.bmp','.webp','.tif','.tiff'}
CLASSES=['Acne','Fungal_Infection','Hyperpigmentation','Normal_Skin','Skin_Cancer','Vitiligo']
CLASS_TO_ID={x:i for i,x in enumerate(CLASSES)}
FOLDER_NAMES={'acne':'Acne','fungal_infection':'Fungal_Infection','hyper_pigmentation':'Hyperpigmentation',
              'hyperpigmentation':'Hyperpigmentation','normal_skin':'Normal_Skin','normal skin':'Normal_Skin',
              'fungal infection':'Fungal_Infection','skin_cancer':'Skin_Cancer','skin cancer':'Skin_Cancer',
              'vitiligo':'Vitiligo'}
MIX_FRACTIONS={'100_100':(1.,1.),'70_30':(.70,.30),'30_70':(.30,.70)}
assert set(MIXES_TO_RUN)<=set(MIX_FRACTIONS) and len(MIXES_TO_RUN)==len(set(MIXES_TO_RUN))
COMMON={name:ROOT/name for name in ['config','dataset','audits','splits','comparison','figures','logs','private_review']}
for p in COMMON.values():p.mkdir(parents=True,exist_ok=True)
logger=logging.getLogger('dermareal_e3');logger.setLevel(logging.INFO);logger.propagate=False
for h in list(logger.handlers):logger.removeHandler(h);h.close()
for h in [logging.FileHandler(COMMON['logs']/'run.log',encoding='utf-8'),logging.StreamHandler(sys.stdout)]:
    h.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'));logger.addHandler(h)
def save_json(obj,path):
    p=Path(path);p.parent.mkdir(parents=True,exist_ok=True)
    with p.open('w',encoding='utf-8') as f:json.dump(obj,f,indent=2,default=str,allow_nan=False)
def find_class_root(hint, source):
    if hint is None:raise FileNotFoundError('Set dataset hint explicitly')
    base=Path(hint)
    if not base.exists():raise FileNotFoundError(f'Missing Kaggle input: {base}. Attach the dataset under Add Input.')
    candidates=[base]+[x for x in base.glob('*') if x.is_dir()]
    candidates += [x for x in base.glob('*/*') if x.is_dir()]
    if base == Path('/kaggle/input'):
        candidates += [x for x in base.rglob('*') if x.is_dir()]
    for d in candidates:
        names={x.name.casefold() for x in d.iterdir() if x.is_dir()}
        labels={FOLDER_NAMES.get(x) for x in names}
        if labels==set(CLASSES) and len(names)==6:
            if source=='real' and 'skin_cancer' not in names:continue
            if source=='synthetic' and 'skin cancer' not in names:continue
            return d
    raise FileNotFoundError(f'Six class folders not found within {base}; check INPUT_HINT')
def discover_input_files(base):
    if not base.exists():return []
    return sorted(p for p in base.rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTS)
REAL_ROOT=find_class_root(REAL_HINT,'real')
SYNTH_ROOT=find_class_root(SYNTH_HINT,'synthetic')
assert SYNTH_ROOT.resolve()!=REAL_ROOT.resolve(),'Real and synthetic roots must differ'
logger.info('Real dataset: %s',REAL_ROOT);logger.info('Synthetic dataset: %s',SYNTH_ROOT)
save_json({'experiment':'E3','seed':SEED,'classes':CLASSES,'real_root':str(REAL_ROOT),'synthetic_root':str(SYNTH_ROOT),
    'mix_fractions':MIX_FRACTIONS,'mixes_selected':MIXES_TO_RUN,'requested_models':4,
    'epochs':EPOCHS,'learning_rates':LEARNING_RATES,'image_size':IMG_SIZE,'batch':BATCH_SIZE,
    'phash_distance':PHASH_DIST,'embed_link_tau':EMBED_LINK_TAU,'audit_thresholds':AUDIT_THRESHOLDS,
    'label_review':'manual diagnostic validation is not replaced by code','e2_frozen_split_required':False,
    'real_split_mode':'self_computed_E2_algorithm', 'synthetic_split_mode':'prompt_group_and_visual_similarity', 'split_mode':SPLIT_MODE, 'real_frozen_hint':str(REAL_FROZEN_SPLIT_HINT) if REAL_FROZEN_SPLIT_HINT else None, 'synthetic_frozen_hint':str(E3_FROZEN_SPLIT_HINT) if E3_FROZEN_SPLIT_HINT else None},
    COMMON['config']/'run_config.json')
save_json({'python':sys.version,'tensorflow':tf.__version__,'numpy':np.__version__,
    'pandas':pd.__version__,'sklearn':__import__('sklearn').__version__,
    'pillow':__import__('PIL').__version__,'platform':platform.platform(),
    'gpu':[str(x) for x in tf.config.list_physical_devices('GPU')]},COMMON['config']/'environment.json')

## 2. Validate corrected real images, group visual duplicates and freeze the real split

Validates image decoding, file hashes, and image dimensions; groups identical images, flip-aware pHash neighbors, ResNet-50 feature neighbors and known patient IDs (only when provided); quarantines cross-class conflicts and makes a 70/15/15 group-aware split. It outputs `dataset/dataset_manifest.csv`, `splits/splits_v3.csv`, grouping CSVs and removal reports. A saved real split is **optional on the first run**, but required together with the synthetic split when reusing another account's assignments. The notebook does not infer medical diagnoses or patient identities.

In [ ]:
# E3 self-contained real-data split — based directly on the corrected E2 notebook.
# This does NOT require any CSV input. It writes new CSV evidence automatically.
# Reproducible within this protocol, but equivalence to an ALREADY EXECUTED E2
# split cannot be asserted unless the saved E2 manifests are compared.
DATA_DIR=REAL_ROOT
EXPECTED_FOLDERS={p.name:FOLDER_NAMES[p.name.casefold()] for p in sorted(REAL_ROOT.iterdir()) if p.is_dir()}
assert set(EXPECTED_FOLDERS.values())==set(CLASSES) and len(EXPECTED_FOLDERS)==len(CLASSES)
DIRS=COMMON
json_save=save_json
SOURCE_MANIFEST_CSV=None   # optional E2-compatible provenance CSV, if independently available
# The first account creates these from images. Other accounts supply the frozen assignment,
# without changing the underlying E2-style group-aware splitting algorithm.
EXISTING_SPLIT_CSV=REAL_FROZEN_SPLIT_HINT
if EXISTING_SPLIT_CSV and str(EXISTING_SPLIT_CSV).lower().endswith('.zip'):
    _fp=COMMON['splits']/'_imported_real_split.csv'
    with zipfile.ZipFile(EXISTING_SPLIT_CSV) as z:
        k='splits/splits_v3.csv'
        if k not in z.namelist():raise FileNotFoundError(f'{k} not found inside {EXISTING_SPLIT_CSV}')
        _fp.write_bytes(z.read(k))
    EXISTING_SPLIT_CSV=str(_fp)
ALLOW_SUSPECT_FILENAMES=False
REQUIRE_PROVENANCE=False
PHASH_MAX_DIST=PHASH_DIST
SIM_TAU_START, SIM_TAU_MAX, SIM_TAU_STEP = .90, .98, .02
MAX_GROUP_SHARE=.02
SPLIT_TRIES=200
CHUNK=192
IMG_SIZE=224
logger.info('Computing real-image E2-compatible grouping directly from %s',DATA_DIR)

SUSPECT = {
    'Skin_Cancer':['sjs-ten','sjs_ten','stevens-johnson','toxic-epidermal','toxic_epidermal'],
    'Normal_Skin':['rosacea','dermatitis','eczema','perioral','seborrheic','psoriasis','vitiligo','melanoma','carcinoma']}
records=[]; invalid=[]; suspicious=[]; total_candidates=0
for folder,cls in EXPECTED_FOLDERS.items():
    files=sorted(p for p in (DATA_DIR/folder).rglob('*') if p.is_file() and p.suffix.lower() in VALID_EXTS)
    total_candidates+=len(files)
    logger.info('%s: %s candidate files',folder,len(files))
    for path in files:
        relative=path.relative_to(DATA_DIR).as_posix()
        try:
            sha=hashlib.sha256(); md5=hashlib.md5()
            with path.open('rb') as f:
                for block in iter(lambda:f.read(1<<20),b''):
                    sha.update(block); md5.update(block)
            with Image.open(path) as img:
                img.load(); w,h=img.size
                if min(w,h)<64: raise ValueError('image smaller than 64 pixels')
                img=ImageOps.exif_transpose(img).convert('RGB')
                small=np.array(img.resize((64,64),Image.Resampling.BILINEAR),dtype=np.float32)
                std=float(small.std()); dark=float((small<4).all(axis=2).mean())
                if std<3 or dark>0.97: raise ValueError('blank/black image')
                border=np.concatenate([small[:8].reshape(-1,3),small[-8:].reshape(-1,3),
                    small[:, :8].reshape(-1,3),small[:, -8:].reshape(-1,3)],axis=0)
                bmean=border.mean(axis=0); bstd=border.std(axis=0)
            item={'relative_path':relative,'path':str(path),'folder':folder,'class':cls,'label':CLASS_TO_ID[cls],
                  'sha256':sha.hexdigest(),'md5':md5.hexdigest(),'width':w,'height':h,'aspect_ratio':round(w/h,6),
                  'size_bytes':path.stat().st_size,'format':path.suffix.lower(),
                  'pixel_std':round(std,3),'dark_fraction':round(dark,4),
                  **{f'border_mean_{k}':float(bmean[i]) for i,k in enumerate('rgb')},
                  **{f'border_std_{k}':float(bstd[i]) for i,k in enumerate('rgb')}}
            records.append(item)
            matches=[word for word in SUSPECT.get(cls,[]) if word in path.name.lower()]
            if matches: suspicious.append({'relative_path':relative,'class':cls,'matched_tokens':';'.join(matches)})
        except Exception as e: invalid.append({'relative_path':relative,'reason':repr(e)})

manifest=pd.DataFrame(records).sort_values('relative_path').reset_index(drop=True)
assert len(manifest)>0, 'No usable images found'
pd.DataFrame(invalid,columns=['relative_path','reason']).to_csv(DIRS['dataset']/'invalid_images.csv',index=False)
pd.DataFrame(suspicious,columns=['relative_path','class','matched_tokens']).to_csv(DIRS['dataset']/'suspect_filename_labels.csv',index=False)

decoded_before_provenance=len(manifest)
# Optional fully traceable provenance. NEVER guess patient IDs or original labels.
manifest['source_dataset']='unknown'; manifest['source_url']=''; manifest['patient_id']=''
manifest['original_diagnosis']=''; manifest['review_status']='unverified'
manifest['license']='unknown'; manifest['allowed_for_training']='unknown'
if SOURCE_MANIFEST_CSV:
    origin=pd.read_csv(SOURCE_MANIFEST_CSV,dtype=str).fillna('')
    assert 'relative_path' in origin.columns and origin.relative_path.is_unique
    fields=['source_dataset','source_url','patient_id','original_diagnosis','review_status','license','allowed_for_training']
    for name in fields:
        if name in origin:
            value_map=origin.set_index('relative_path')[name]
            manifest[name]=manifest.relative_path.map(value_map).fillna('')
    reject=manifest.allowed_for_training.str.lower().isin(['false','no','0','not_allowed'])
    reject|=manifest.review_status.str.lower().isin(['reject','rejected','incorrect_label'])
    if reject.any():
        manifest.loc[reject,['relative_path','class','sha256','md5','source_dataset',
            'original_diagnosis','review_status','license','allowed_for_training']].to_csv(
                DIRS['dataset']/'provenance_excluded.csv',index=False)
        manifest=manifest.loc[~reject].copy().reset_index(drop=True)
    logger.info('Provenance manifest merged: %s',SOURCE_MANIFEST_CSV)

provenance_removed=decoded_before_provenance-len(manifest)
if not (DIRS['dataset']/'provenance_excluded.csv').exists():
    pd.DataFrame(columns=['relative_path','class','sha256','md5','source_dataset',
        'original_diagnosis','review_status','license','allowed_for_training']).to_csv(
        DIRS['dataset']/'provenance_excluded.csv',index=False)
# Patient identifiers must NOT be exported to a public research repository.
# Use a private ephemeral HMAC key (or DERMA_PATIENT_HMAC_KEY from Kaggle Secrets).
# Raw IDs are discarded; equality is retained for grouping within this run.
import hmac, secrets
_private_patient_key = os.environ.get('DERMA_PATIENT_HMAC_KEY', '').encode('utf-8') or secrets.token_bytes(32)
manifest['patient_id'] = [
    hmac.new(_private_patient_key, (str(src)+'|'+str(pid)).encode('utf-8'), hashlib.sha256).hexdigest()[:32]
    if str(pid).strip() and str(pid).lower() != 'nan' else ''
    for src,pid in zip(manifest['source_dataset'],manifest['patient_id'])
]
del _private_patient_key

suspicious_active=[x for x in suspicious if x['relative_path'] in set(manifest.relative_path)]
manifest['status']='candidate'
manifest.to_csv(DIRS['dataset']/'dataset_manifest.csv',index=False)
counts=manifest.groupby('class').size().reindex(CLASSES,fill_value=0).rename('usable_count').reset_index()
counts.to_csv(DIRS['dataset']/'dataset_statistics.csv',index=False)
logger.info('Usable image counts:\n%s',counts.to_string(index=False))
logger.info('Invalid=%s; suspicious filenames=%s; clinical labels NOT automatically validated',len(invalid),len(suspicious))
if REQUIRE_PROVENANCE:
    assert (manifest.source_dataset.fillna('').str.strip().ne('') & manifest.source_dataset.ne('unknown')).all() and (manifest.license.fillna('').str.strip().ne('') & manifest.license.ne('unknown')).all(), 'Missing provenance/licence'
if suspicious_active and not ALLOW_SUSPECT_FILENAMES:
    raise RuntimeError('Known wrong-label filename patterns detected. Review suspect_filename_labels.csv; correct/remove the files, then rerun. Override only after documented verification.')
assert (counts.usable_count>5).all(), 'A class has too few valid files.'

# Dataset content fingerprint: different data cannot silently reuse a split.
dataset_fingerprint=hashlib.sha256('\n'.join(manifest.relative_path+'|'+manifest.sha256+'|'+manifest['class']).encode()).hexdigest()
json_save({'fingerprint_sha256':dataset_fingerprint,'files':len(manifest),'invalid':len(invalid),
           'total_candidate_images':total_candidates,'valid_decoded_images':decoded_before_provenance,
           'removed_by_provenance':provenance_removed,
           'suspect_filename_count':len(suspicious), 'suspect_filename_still_active':len(suspicious_active),'clinical_label_verification':'NOT INFERRED'},
          DIRS['dataset']/'dataset_fingerprint.json')
logger.info('Dataset fingerprint: %s',dataset_fingerprint)
json_save({'candidate_files':total_candidates,'invalid_images':len(invalid),
           'decoded_images':decoded_before_provenance,'provenance_exclusions':provenance_removed,
           'eligible_for_grouping':len(manifest),'suspicious_active_filenames':len(suspicious_active)},
          DIRS['dataset']/'validation_summary.json')
class UnionFind:
    def __init__(self,n): self.parent=np.arange(n,dtype=np.int32); self.rank=np.zeros(n,dtype=np.int8)
    def find(self,x):
        p=self.parent
        while p[x]!=x: p[x]=p[p[x]]; x=p[x]
        return int(x)
    def union(self,x,y):
        a,b=self.find(int(x)),self.find(int(y))
        if a==b: return
        if self.rank[a]<self.rank[b]: a,b=b,a
        self.parent[b]=a
        if self.rank[a]==self.rank[b]: self.rank[a]+=1
    def groups(self):
        roots=np.array([self.find(i) for i in range(len(self.parent))])
        return pd.factorize(roots,sort=True)[0].astype(np.int32)

N=len(manifest); uf_exact=UnionFind(N)
for _,indices in manifest.groupby('sha256').indices.items():
    ii=list(indices)
    for j in ii[1:]: uf_exact.union(ii[0],j)
manifest['exact_group']=uf_exact.groups()
exact_g=manifest.groupby('sha256').agg(image_count=('path','size'),n_classes=('class','nunique'))
exact_g.to_csv(DIRS['audits']/'exact_sha256_groups.csv')
md5_g=manifest.groupby('md5').agg(image_count=('path','size'),distinct_sha256=('sha256','nunique'),n_classes=('class','nunique'))
md5_g.to_csv(DIRS['audits']/'exact_md5_groups.csv')
if (md5_g.distinct_sha256>1).any():
    raise RuntimeError('MD5 collision or inconsistent hashing detected; investigate before splitting')
logger.info('Exact content: %s repeated-hash groups, %s extra copies, %s conflicting hashes',
            int((exact_g.image_count>1).sum()),int((exact_g.image_count-1).sum()),int((exact_g.n_classes>1).sum()))

from PIL import ImageOps

def phash64(image):
    # Equivalent 32x32 DCT / 8x8 median algorithm to ImageHash.phash(hash_size=8).
    gray=image.convert('L').resize((32,32),Image.Resampling.LANCZOS)
    pixels=np.asarray(gray,dtype=np.float32)
    coeff=dct(dct(pixels,axis=0),axis=1)
    lo=coeff[:8,:8]; bits=(lo>np.median(lo)).reshape(-1)
    value=0
    for bit in bits: value=(value<<1)|int(bit)
    return value

def hash_variants(path):
    with Image.open(path) as raw:
        im=ImageOps.exif_transpose(raw).convert('RGB')
        variants=[im,ImageOps.mirror(im),ImageOps.flip(im),ImageOps.mirror(ImageOps.flip(im))]
        return [phash64(a) for a in variants]

HV=np.asarray([hash_variants(p) for p in manifest.path],dtype=np.uint64)
manifest['phash_original']=[f'{int(x):016x}' for x in HV[:,0]]
POP=np.array([int(i).bit_count() for i in range(256)],dtype=np.uint8)
def hamming_block(a,b):
    xor=np.bitwise_xor(a[:,None],b[None,:])
    bytes_=np.ascontiguousarray(xor).view(np.uint8).reshape(len(a),len(b),8)
    return POP[bytes_].sum(axis=2,dtype=np.uint8)

uf_phash=UnionFind(N)
plain_pairs=0; flipped_pairs=0
for start in range(0,N,CHUNK):
    stop=min(N,start+CHUNK)
    d0=hamming_block(HV[start:stop,0],HV[:,0]); d=d0.copy()
    for k in range(1,4):
        np.minimum(d,hamming_block(HV[start:stop,0],HV[:,k]),out=d)
        np.minimum(d,hamming_block(HV[start:stop,k],HV[:,0]),out=d)
    row,col=np.nonzero(d<=PHASH_MAX_DIST); row=row+start
    keep=col>row; row,col=row[keep],col[keep]
    for a,b in zip(row,col): uf_phash.union(a,b)
    flipped_pairs+=len(row)
    row2,col2=np.nonzero(d0<=PHASH_MAX_DIST); row2=row2+start
    plain_pairs+=int((col2>row2).sum())
manifest['phash_group']=uf_phash.groups()
logger.info('pHash: plain linked pairs %d; flip-aware %d; flip-aware groups %d',plain_pairs,flipped_pairs,manifest.phash_group.nunique())

# Embed images with same aspect-preserving letterbox used in the training pipeline.
def load_rgb(path,size=224):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im=ImageOps.pad(im,(size,size),method=Image.Resampling.BILINEAR,color=(127,127,127))
        return np.asarray(im,dtype=np.float32)

def embed_images(model_name, batch_size=32):
    if model_name=='resnet':
        net=ResNet50(include_top=False,weights='imagenet',pooling='avg',input_shape=(IMG_SIZE,IMG_SIZE,3))
    else:
        net=EfficientNetV2B0(include_top=False,weights='imagenet',pooling='avg',
                             input_shape=(IMG_SIZE,IMG_SIZE,3),include_preprocessing=True)
    a,b=[],[]
    for start in range(0,N,batch_size):
        arr=np.stack([load_rgb(p,IMG_SIZE) for p in manifest.path.iloc[start:start+batch_size]])
        flipped=arr[:,:,::-1,:].copy()
        if model_name=='resnet':
            arr=keras.applications.resnet50.preprocess_input(arr)
            flipped=keras.applications.resnet50.preprocess_input(flipped)
        a.append(net.predict(arr,verbose=0)); b.append(net.predict(flipped,verbose=0))
    del net; gc.collect(); keras.backend.clear_session()
    return np.concatenate(a).astype(np.float32),np.concatenate(b).astype(np.float32)

def normalized_embeddings(a,b):
    mu=a.mean(axis=0,keepdims=True)
    a=a-mu; b=b-mu
    a/=np.linalg.norm(a,axis=1,keepdims=True)+1e-8
    b/=np.linalg.norm(b,axis=1,keepdims=True)+1e-8
    return a,b

logger.info('Calculating ResNet-50 grouping embeddings for %s images',N)
E,Ef=normalized_embeddings(*embed_images('resnet'))
# Candidate links; store similarities so tau can be tightened without re-embedding.
ei,ej,es=[],[],[]
for start in range(0,N,CHUNK):
    sim=np.maximum(E[start:start+CHUNK]@E.T,E[start:start+CHUNK]@Ef.T)
    ii,jj=np.nonzero(sim>=SIM_TAU_START); ii=ii+start
    take=jj>ii
    ii,jj=ii[take],jj[take]
    if len(ii): ei.append(ii.astype(np.int32));ej.append(jj.astype(np.int32));es.append(sim[(ii-start),jj].astype(np.float32))
ei=np.concatenate(ei) if ei else np.array([],dtype=np.int32)
ej=np.concatenate(ej) if ej else np.array([],dtype=np.int32)
es=np.concatenate(es) if es else np.array([],dtype=np.float32)
del E,Ef; gc.collect()

# Base pHash grouping + exact file hashes + real patient IDs (if present).
def make_groups(tau):
    u=UnionFind(N)
    for col in ['phash_group','exact_group']:
        for _,inds in manifest.groupby(col).indices.items():
            ind=list(inds)
            for j in ind[1:]:u.union(ind[0],j)
    known=manifest[manifest.patient_id.fillna('').str.strip().ne('') &
                   manifest.source_dataset.fillna('').str.strip().ne('unknown')]
    for _,inds in known.groupby(['source_dataset','patient_id']).indices.items():
        ind=list(known.index[inds])
        for j in ind[1:]:u.union(ind[0],j)
    for x,y in zip(ei[es>=tau],ej[es>=tau]):u.union(x,y)
    return u.groups()

tau=SIM_TAU_START
while True:
    groups=make_groups(tau)
    group_sizes=pd.Series(groups).value_counts()
    if group_sizes.max()/N<=MAX_GROUP_SHARE or tau>=SIM_TAU_MAX-1e-8: break
    tau=round(min(tau+SIM_TAU_STEP,SIM_TAU_MAX),2)
manifest['group']=groups
logger.info('Final groups=%d; tau=%.2f; largest group=%d (%.2f%%)',
            manifest.group.nunique(),tau,group_sizes.max(),100*group_sizes.max()/N)
if group_sizes.max()/N>MAX_GROUP_SHARE:
    logger.warning('Largest group exceeds group-share target; inspect group_summary.csv')

stats=manifest.groupby('group').agg(group_size=('path','size'),n_classes=('class','nunique'),
                                    classes=('class',lambda s:';'.join(sorted(s.unique()))),
                                    distinct_sha=('sha256','nunique'))
stats.to_csv(DIRS['audits']/'group_summary.csv')
conflict_ids=stats.index[stats.n_classes>1].tolist()
manifest.loc[manifest.group.isin(conflict_ids)].to_csv(DIRS['audits']/'cross_class_conflicts.csv',index=False)
logger.warning('%d cross-class visual groups quarantined pending manual review',len(conflict_ids))
# Quarantine ALL members of conflicting-class groups to avoid label-based leakage.
manifest['status']='eligible';manifest.loc[manifest.group.isin(conflict_ids),'status']='quarantined_cross_class'
# Mark exact-byte cross-label conflicts independently for the MD5 evidence table.
exact_conflict_hashes=set(exact_g.index[exact_g.n_classes>1])
manifest.loc[manifest.sha256.isin(exact_conflict_hashes),'status']='quarantined_cross_class_exact' 
# Keep ONE identical-file copy per class. All same-group related but non-identical images are retained.
dup_mask=(manifest.status=='eligible') & manifest.duplicated('sha256',keep='first')
manifest.loc[dup_mask,'status']='removed_exact_duplicate'
manifest.to_csv(DIRS['dataset']/'dataset_manifest.csv',index=False)
manifest.to_csv(DIRS['audits']/'group_membership.csv',index=False)
manifest.loc[manifest.status!='eligible'].to_csv(DIRS['audits']/'excluded_images.csv',index=False)
clean=manifest.loc[manifest.status=='eligible'].copy().reset_index(drop=True)
assert clean.groupby('class').size().reindex(CLASSES,fill_value=0).min()>10,'A class too small after quarantine'
report={'original_files':total_candidates,'valid_after_decode':decoded_before_provenance,
        'after_provenance_exclusions':N,'invalid_files':len(invalid),
        'provenance_excluded':provenance_removed,'final_eligible':len(clean),
        'exact_hash_groups_repeated':int((exact_g.image_count>1).sum()),
        'extra_exact_copies_total':int((exact_g.image_count-1).sum()),
        'exact_duplicate_copies_removed':int(dup_mask.sum()),
        'cross_class_groups_quarantined':len(conflict_ids),
        'cross_class_images_quarantined':int(manifest.status.str.startswith('quarantined_cross_class').sum()),
        'unique_visual_groups':int(manifest.group.nunique()),
        'eligible_visual_groups':int(clean.group.nunique()),
        'multi_image_groups':int((stats.group_size>1).sum()),
        'largest_group':int(stats.group_size.max()),
        'plain_phash_pairs':plain_pairs,'flip_aware_phash_pairs':flipped_pairs,
        'embedding_tau':tau,'patient_grouping_used':bool((manifest.patient_id!='').any()),
        'group_type':'visual similarity; known patient ID when supplied; not inferred patient identity'}
json_save(report,DIRS['audits']/'grouping_summary.json')
class_stats=manifest.groupby(['class','status']).size().rename('count').reset_index()
class_stats.to_csv(DIRS['audits']/'removals_by_class_reason.csv',index=False)
manifest.groupby(['source_dataset','status'],dropna=False).size().rename('image_count').reset_index().to_csv(
    DIRS['audits']/'removals_by_source_reason.csv',index=False)
manifest.loc[manifest.status!='eligible',
    ['relative_path','class','source_dataset','sha256','md5','group','status']].to_csv(
    DIRS['audits']/'excluded_image_details.csv',index=False)
# PRIVATE gallery for diagnosis review; this may include copyrighted or identifiable images.
# It is deliberately kept OUT of the GitHub evidence ZIP.
PRIVATE_REVIEW_DIR=ROOT/'private_review'
PRIVATE_REVIEW_DIR.mkdir(parents=True,exist_ok=True)
if conflict_ids:
    sample_ids=conflict_ids[:min(20,len(conflict_ids))]
    for group_id in sample_ids:
        group_rows=manifest.loc[manifest.group==group_id].head(8)
        fig,axes=plt.subplots(2,4,figsize=(12,7))
        for ax in axes.ravel():ax.axis('off')
        for ax,(_,rec) in zip(axes.ravel(),group_rows.iterrows()):
            ax.imshow(load_rgb(rec['path'],160).astype(np.uint8))
            ax.set_title(f"{rec['class']}\n{Path(rec['path']).name[:30]}",fontsize=7)
        fig.suptitle(f'PRIVATE — conflicting visual group {group_id}: manual review required')
        fig.tight_layout()
        fig.savefig(PRIVATE_REVIEW_DIR/f'conflict_group_{group_id}.png',dpi=110)
        plt.close(fig)
print(json.dumps(report,indent=2))
def best_group_split(y,g,test_fraction,tries,seed):
    y=np.asarray(y,dtype=int);g=np.asarray(g)
    target=np.bincount(y,minlength=len(CLASSES))/len(y)
    best=None
    for i in range(tries):
        splitter=GroupShuffleSplit(n_splits=1,test_size=test_fraction,random_state=seed+i)
        a,b=next(splitter.split(np.zeros(len(y)),y,groups=g))
        if not len(a) or not len(b):continue
        counts=np.bincount(y[b],minlength=len(CLASSES)); prop=counts/len(b)
        score=float(np.max(np.abs(prop-target)/(target+1e-8))+2*abs(len(b)/len(y)-test_fraction))
        if best is None or score<best[0]:best=(score,a,b)
    if best is None:raise ValueError('Group split failed; insufficient independent groups')
    return best[1],best[2]

if EXISTING_SPLIT_CSV:
    old=pd.read_csv(EXISTING_SPLIT_CSV,dtype={'sha256':str,'relative_path':str,'class':str,'split':str})
    assert set(old.columns)>={'relative_path','sha256','class','split','group','dataset_fingerprint'}
    assert old.relative_path.is_unique and set(old.relative_path)==set(clean.relative_path)
    assert old['split'].isin(['train','val','test']).all()
    assert set(old.dataset_fingerprint)=={dataset_fingerprint},'Dataset fingerprint mismatch: do not reuse split'
    tmp=clean[['relative_path','sha256','class','group']].merge(old[['relative_path','sha256','class','split']],on='relative_path',suffixes=('_actual','_saved'),validate='one_to_one')
    assert (tmp.sha256_actual==tmp.sha256_saved).all() and (tmp.class_actual==tmp.class_saved).all()
    clean['split']=clean.relative_path.map(old.set_index('relative_path')['split'])
    assert clean.groupby('group').split.nunique().max()==1, 'Frozen real split separates a visual/patient group'
    logger.info('Reusing strictly checked existing split: %s',EXISTING_SPLIT_CSV)
else:
    tr,temp=best_group_split(clean.label,clean.group,0.30,SPLIT_TRIES,SEED)
    va_sub,te_sub=best_group_split(clean.label.iloc[temp],clean.group.iloc[temp],0.50,SPLIT_TRIES,SEED+3000)
    split=np.empty(len(clean),dtype=object)
    split[tr]='train'; split[temp[va_sub]]='val'; split[temp[te_sub]]='test'
    clean['split']=split

assert clean.split.isin(['train','val','test']).all()
assert clean.groupby('group').split.nunique().max()==1,'Group leakage!'
assert clean.groupby('sha256').split.nunique().max()==1,'SHA-256 leakage!'
assert all(clean.loc[clean.split==s,'class'].nunique()==len(CLASSES) for s in ['train','val','test']), 'A split lacks one of six classes'
clean['dataset_fingerprint']=dataset_fingerprint
export_cols=['relative_path','sha256','class','label','group','split','dataset_fingerprint',
             'source_dataset','patient_id','review_status']
clean[export_cols].to_csv(DIRS['splits']/'splits_v3.csv',index=False)
# Preserve absolute mount path separately for this run; E3 should remap on relative_path, not on mount prefix.
clean.to_csv(DIRS['splits']/'split_manifest.csv',index=False)
counts=pd.crosstab(clean['class'],clean['split']).reindex(index=CLASSES,columns=['train','val','test'],fill_value=0)
counts.to_csv(DIRS['splits']/'split_class_counts.csv')
ng=clean.groupby(['split','class']).group.nunique().rename('visual_groups').reset_index()
ng.to_csv(DIRS['splits']/'split_group_counts.csv',index=False)
logger.info('Frozen counts:\n%s',counts.to_string())
json_save({'fingerprint':dataset_fingerprint,'group_overlap':0,'sha256_overlap':0,
           'split_counts':clean.split.value_counts().to_dict(),
           'split_csv':'splits/splits_v3.csv'},DIRS['splits']/'split_assertions.json')

# Publication figure 1 & figure 4 are newly generated from THIS dataset, not copied from old run.
fig,ax=plt.subplots(figsize=(10,4)); counts.sum(axis=1).plot.bar(ax=ax,rot=25)
ax.set_title('Corrected real facial skin dataset: usable images by class');ax.set_ylabel('Files after dedup/quarantine')
fig.tight_layout();fig.savefig(DIRS['figures']/'fig1_corrected_dataset_distribution.png',dpi=220);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,4));counts.plot.bar(ax=ax,rot=25)
ax.set_title('Frozen real-data group-aware train/validation/test counts');ax.set_ylabel('Images')
fig.tight_layout();fig.savefig(DIRS['figures']/'fig4_real_split.png',dpi=220);plt.close(fig)

# Convert E2's self-computed manifest/split into E3's expected schema.
real=clean.copy()
real['path']=real.relative_path.map(lambda rp:str(REAL_ROOT/rp))
real['label']=real['class'].map(CLASS_TO_ID).astype(int)
real['source']='real'
real['record_id']='real:'+real.relative_path
real['group_key']='real:'+real.group.astype(str)
E2_ORIGIN='self_computed_from_dataset_no_external_E2_artifacts'
assert real.groupby('group').split.nunique().max()==1
assert real.groupby('sha256').split.nunique().max()==1
real.drop(columns=[c for c in ['patient_id'] if c in real]).to_csv(
    COMMON['splits']/'real_self_computed.csv',index=False)
save_json({'origin':E2_ORIGIN,'dataset_fingerprint':dataset_fingerprint,
    'n_real':int(len(real)),'group_overlap':0,'hash_overlap':0,
    'is_identical_to_previously_completed_E2_split':'UNVERIFIED',
    'warning':'If E2 ran with a different provenance manifest or grouping, test sets may differ.'},
    COMMON['splits']/'real_split_provenance.json')
logger.info('Real-data split recomputed without external CSV: %s',real.split.value_counts().to_dict())
# Restore synthetic processing settings for the remaining E3 cells.
CHUNK=160


## 3. Synthetic dataset inventory and prompt-ID validation

Parses the original diffusion filename suffix `_p_<prompt>_v_<variation>` and GAN subtype suffixes `_BCC_`, `_SCC_`, `_MEL_`; records SHA-256, MD5, validity, and class mapping. Exact copies, cross-class conflicts and incomplete five-variant prompt groups are reported or excluded before splitting. Images remain traceable to their real or synthetic origin.

In [ ]:
PROMPT_RE=re.compile(r'_p_(\d+)_v_(\d+)$',re.I)
CANCER_RE=re.compile(r'_(BCC|SCC|MEL)_(\d+)$',re.I)
def load_rgb(path,size=IMG_SIZE):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        im=ImageOps.pad(im,(size,size),method=Image.Resampling.BILINEAR,color=(127,127,127))
        return np.asarray(im,dtype=np.float32)
records=[];invalid=[]
for p in discover_input_files(SYNTH_ROOT):
    rel=p.relative_to(SYNTH_ROOT).as_posix(); f=rel.split('/')[0].casefold()
    cls=FOLDER_NAMES.get(f)
    if cls is None:raise ValueError(f'Unknown synthetic class folder {f}')
    try:
        sha,md5=hashlib.sha256(),hashlib.md5()
        with p.open('rb') as fobj:
            for block in iter(lambda:fobj.read(1<<20),b''):
                sha.update(block);md5.update(block)
        with Image.open(p) as im:
            im.load();w,h=im.size
            if min(w,h)<64:raise ValueError('small image <64px')
            arr=np.asarray(ImageOps.exif_transpose(im).convert('RGB').resize((64,64)),dtype=np.float32)
            if arr.std()<3 or np.mean(np.all(arr<4,axis=-1))>0.97:raise ValueError('degenerate/black image')
        stem=p.stem;m=PROMPT_RE.search(stem);mc=CANCER_RE.search(stem)
        prompt_id=m.group(1) if m else ''
        subtype=mc.group(1).upper() if mc else ''
        group_hint='prompt:'+cls+':'+str(int(prompt_id)) if m else ('gan:'+subtype if mc else 'unknown:'+cls)
        records.append({'source':'synthetic','relative_path':rel,'path':str(p),'class':cls,
            'label':CLASS_TO_ID[cls],'sha256':sha.hexdigest(),'md5':md5.hexdigest(),'width':w,'height':h,
            'size_bytes':p.stat().st_size,'aspect_ratio':w/h,'prompt_id':prompt_id,'gan_subtype':subtype,
            'group_hint':group_hint,'status':'candidate'})
    except Exception as e:invalid.append({'relative_path':rel,'reason':repr(e)})
synth_all=pd.DataFrame(records).sort_values('relative_path').reset_index(drop=True)
assert not synth_all.empty and set(synth_all['class'])==set(CLASSES),'Missing synthetic classes'
pd.DataFrame(invalid,columns=['relative_path','reason']).to_csv(COMMON['dataset']/'synthetic_invalid_images.csv',index=False)
synth_all['status']='eligible'
# A synthetic image matching ANY real split is excluded, never deleted from real.
real_hashes=set(real.sha256)
synth_all.loc[synth_all.sha256.isin(real_hashes),'status']='removed_shared_exact_real'
conflict_sha=set(synth_all.groupby('sha256')['class'].nunique().loc[lambda z:z>1].index)
synth_all.loc[synth_all.sha256.isin(conflict_sha),'status']='quarantined_synthetic_label_conflict'
mask=(synth_all.status=='eligible') & synth_all.duplicated('sha256',keep='first')
synth_all.loc[mask,'status']='removed_synthetic_exact_copy'
# Match the original notebook: each diffusion prompt requires 5 distinct variants.
# Run this after exact-content quarantine; partial prompt groups are not split.
prompt_rows=synth_all[(synth_all.prompt_id!='')].copy()
prompt_report=[]
for (cls,pid),part in prompt_rows.groupby(['class','prompt_id'],sort=True):
    eligible=part.loc[part.status=='eligible']
    nums=[]
    for rp in eligible.relative_path:
        mm=PROMPT_RE.search(Path(rp).stem)
        nums.append(int(mm.group(2)) if mm else -1)
    complete=(len(eligible)==5 and len(set(nums))==5)
    prompt_report.append({'class':cls,'prompt_id':pid,'eligible_variants':len(eligible),
        'distinct_variants':len(set(nums)),'complete_five_variant_group':complete})
    if not complete:
        synth_all.loc[part.index[synth_all.loc[part.index,'status']=='eligible'],'status']='quarantined_incomplete_prompt_group'
pd.DataFrame(prompt_report).to_csv(COMMON['audits']/'synthetic_prompt_completeness.csv',index=False)
logger.info('Synthetic: candidates=%d eligible_after_exact=%d invalid=%d shared-real=%d conflicts=%d',
    len(synth_all),sum(synth_all.status=='eligible'),len(invalid),sum(synth_all.status=='removed_shared_exact_real'),len(conflict_sha))
synth_all.groupby(['class','status']).size().rename('n').reset_index().to_csv(COMMON['dataset']/'synthetic_quality_counts.csv',index=False)
synth_all.groupby(['source','status']).size().rename('n').reset_index().to_csv(COMMON['audits']/'duplicate_removals_by_source.csv',index=False)
synth_all[['relative_path','class','sha256','md5','prompt_id','gan_subtype','group_hint','status']].to_csv(COMMON['dataset']/'synthetic_manifest.csv',index=False)
assert (synth_all.status=='eligible').sum()>0

## 4. Prompt-wise synthetic split and visual duplicate protections

All five variations of a diffusion prompt share a single group. The additional flip-aware pHash and ResNet-50 links may merge prompts, so this **extends rather than identically replicates** the old prompt-level random split. Skin-cancer GAN outputs are grouped by visual similarity, with subtype information retained. Write `splits/synthetic_frozen_v1.csv`; import exactly that file in later Kaggle accounts to keep identical assignments. Exact and visual exclusions are saved separately.

In [ ]:
class UnionFind:
    def __init__(self,n):self.p=np.arange(n,dtype=np.int32);self.rank=np.zeros(n,np.int8)
    def find(self,x):
        x=int(x)
        while self.p[x]!=x:self.p[x]=self.p[int(self.p[x])];x=int(self.p[x])
        return x
    def union(self,x,y):
        a,b=self.find(x),self.find(y)
        if a==b:return
        if self.rank[a]<self.rank[b]:a,b=b,a
        self.p[b]=a
        if self.rank[a]==self.rank[b]:self.rank[a]+=1
    def labels(self):return pd.factorize([self.find(i) for i in range(len(self.p))],sort=True)[0]
def phash64(im):
    pix=np.asarray(im.convert('L').resize((32,32),Image.Resampling.LANCZOS),dtype=np.float32)
    coeff=dct(dct(pix,axis=0),axis=1)[:8,:8]
    bits=(coeff>np.median(coeff)).flatten();h=0
    for b in bits:h=(h<<1)|int(b)
    return h
def phash_variants(path):
    with Image.open(path) as im:
        im=ImageOps.exif_transpose(im).convert('RGB')
        return [phash64(a) for a in [im,ImageOps.mirror(im),ImageOps.flip(im),ImageOps.mirror(ImageOps.flip(im))]]
POPCOUNT=np.array([int(x).bit_count() for x in range(256)],dtype=np.uint8)
def hamming(a,b):
    x=np.bitwise_xor(np.asarray(a,dtype=np.uint64)[:,None],np.asarray(b,dtype=np.uint64)[None,:])
    return POPCOUNT[np.ascontiguousarray(x).view(np.uint8).reshape(len(a),len(b),8)].sum(axis=-1)
def all_phashes(paths):return np.array([phash_variants(x) for x in paths],dtype=np.uint64)
def link_phash(uf,arr,threshold):
    n=len(arr);linked=0
    for start in range(0,n,CHUNK):
        stop=min(n,start+CHUNK)
        d=hamming(arr[start:stop,0],arr[:,0])
        for orientation in range(1,4):
            d=np.minimum(d,hamming(arr[start:stop,0],arr[:,orientation]))
            d=np.minimum(d,hamming(arr[start:stop,orientation],arr[:,0]))
        i,j=np.where(d<=threshold);i=i+start;select=j>i
        for x,y in zip(i[select],j[select]):uf.union(int(x),int(y))
        linked+=int(select.sum())
    return linked
def embed_paths(paths,backbone='resnet',batch=32):
    if backbone=='resnet':
        model=ResNet50(include_top=False,weights='imagenet',pooling='avg',input_shape=(IMG_SIZE,IMG_SIZE,3))
    elif backbone=='efficientnet':
        model=EfficientNetV2B0(include_top=False,weights='imagenet',pooling='avg',
             input_shape=(IMG_SIZE,IMG_SIZE,3),include_preprocessing=True)
    else:raise ValueError(backbone)
    orig,flip=[],[]
    for i in range(0,len(paths),batch):
        a=np.stack([load_rgb(p) for p in paths[i:i+batch]])
        b=a[:,:,::-1,:].copy()
        if backbone=='resnet':
            a=keras.applications.resnet50.preprocess_input(a)
            b=keras.applications.resnet50.preprocess_input(b)
        orig.append(model.predict(a,verbose=0));flip.append(model.predict(b,verbose=0))
    del model;gc.collect();keras.backend.clear_session()
    orig=np.concatenate(orig).astype(np.float32);flip=np.concatenate(flip).astype(np.float32)
    mu=orig.mean(axis=0,keepdims=True)
    orig-=mu;flip-=mu
    orig/=np.linalg.norm(orig,axis=1,keepdims=True)+1e-8
    flip/=np.linalg.norm(flip,axis=1,keepdims=True)+1e-8
    return orig,flip

syn=synth_all.loc[synth_all.status=='eligible'].copy().reset_index(drop=True)
assert syn.groupby('class').size().min()>15,'At least one synthetic class too small'
S_HASHES=all_phashes(syn.path.tolist());R_HASHES=all_phashes(real.path.tolist())
uf=UnionFind(len(syn))
# Link repeated prompt variants and identical content before image similarity clustering.
for _,grp in syn.loc[syn.prompt_id!=''].groupby(['class','prompt_id']):
    idx=grp.index.to_list()
    for x in idx[1:]:uf.union(idx[0],x)
for _,ids in syn.groupby('sha256').indices.items():
    idx=list(ids)
    for x in idx[1:]:uf.union(idx[0],x)
phash_pairs=link_phash(uf,S_HASHES,PHASH_DIST)
# Across the two sources, a high-confidence pHash neighbour is excluded on synthetic side.
cross_min=np.full(len(syn),255,dtype=np.int16)
for start in range(0,len(syn),CHUNK):
    stop=min(len(syn),start+CHUNK)
    dist=hamming(S_HASHES[start:stop,0],R_HASHES[:,0])
    for j in range(1,4):
        dist=np.minimum(dist,hamming(S_HASHES[start:stop,0],R_HASHES[:,j]))
        dist=np.minimum(dist,hamming(S_HASHES[start:stop,j],R_HASHES[:,0]))
    cross_min[start:stop]=dist.min(axis=1)
syn['min_phash_to_real']=cross_min
# Embed synthetic to verify additional GAN and cross-prompt near duplicates.
if RUN_SIMILARITY_GROUPING:
    S_EMB,S_EMBF=embed_paths(syn.path.tolist(),'resnet')
    embedding_links=0
    for start in range(0,len(syn),CHUNK):
        scores=np.maximum(S_EMB[start:start+CHUNK]@S_EMB.T,S_EMB[start:start+CHUNK]@S_EMBF.T)
        x,y=np.where(scores>=EMBED_LINK_TAU);x+=start;keep=y>x
        for a,b in zip(x[keep],y[keep]):uf.union(int(a),int(b))
        embedding_links+=int(keep.sum())
    del S_EMB,S_EMBF;gc.collect()
else:embedding_links=0
syn['group']=uf.labels()
conflict_groups=set(syn.groupby('group')['class'].nunique().loc[lambda x:x>1].index)
syn['status']='eligible'
near_real_groups=set(syn.loc[syn.min_phash_to_real<=PHASH_DIST,'group'])
syn.loc[syn.group.isin(conflict_groups),'status']='quarantined_cross_class_group'
syn.loc[syn.group.isin(near_real_groups),'status']='quarantined_near_real_group'
syn.loc[syn.group.isin(conflict_groups & near_real_groups),'status']='quarantined_cross_class_and_real_group'
syn[['relative_path','class','sha256','group','group_hint','min_phash_to_real','status']].to_csv(COMMON['audits']/'synthetic_group_membership.csv',index=False)
syn.groupby('group').agg(group_size=('class','size'),n_classes=('class','nunique'),
    min_real_phash=('min_phash_to_real','min')).to_csv(COMMON['audits']/'synthetic_group_summary.csv')
syn.loc[syn.status!='eligible',['relative_path','class','sha256','group','status']].to_csv(COMMON['audits']/'synthetic_visual_quarantine.csv',index=False)
# Preserve the FINAL status per synthetic image; do not sum overlapping snapshots.
syn_visual_status=syn[['relative_path','status']].set_index('relative_path')['status']
synth_all['final_status']=synth_all['status']
eligible_idx=synth_all['status']=='eligible'
synth_all.loc[eligible_idx,'final_status']=synth_all.loc[eligible_idx,'relative_path'].map(syn_visual_status).fillna('quarantined_missing_visual_record')
assert len(synth_all)==synth_all.final_status.value_counts().sum()
synth_all.groupby(['source','final_status']).size().rename('image_count').reset_index().to_csv(
    COMMON['audits']/'synthetic_final_dispositions.csv',index=False)
syn.groupby(['class','status']).size().rename('n').reset_index().to_csv(
    COMMON['audits']/'visual_quarantine_counts_by_class.csv',index=False)
syn=syn.loc[syn.status=='eligible'].copy().reset_index(drop=True)
assert syn.groupby('class').size().min()>15,'After quarantine, synthetic class insufficient; inspect synthetic_visual_quarantine.csv'
# Freeze group-aware synthetic split once; E2 real split stays unchanged.
syn_fingerprint=hashlib.sha256(('\n'.join((syn.relative_path+'|'+syn.sha256+'|'+syn['class']).sort_values())).encode()).hexdigest()
save_json({'fingerprint_sha256':syn_fingerprint,'clean_images':len(syn),
    'cross_source_phash_quarantined':int(sum(cross_min<=PHASH_DIST)),
    'synthetic_cross_class_groups_quarantined':len(conflict_groups),
    'phash_pairs':phash_pairs,'resnet_embedding_links':embedding_links},COMMON['dataset']/'synthetic_fingerprint.json')
def best_group_split(y,groups,fraction,seed,tries=SPLIT_TRIES):
    y=np.asarray(y,dtype=np.int32);groups=np.asarray(groups)
    freq=np.bincount(y,minlength=len(CLASSES))/len(y)
    best=None
    for k in range(tries):
        gss=GroupShuffleSplit(n_splits=1,test_size=fraction,random_state=seed+k)
        a,b=next(gss.split(np.zeros(len(y)),y,groups=groups))
        if not len(a) or not len(b):continue
        p=np.bincount(y[b],minlength=len(CLASSES))/len(b)
        score=float(np.max(np.abs(p-freq)/(freq+1e-8))+2*abs(len(b)/len(y)-fraction))
        if best is None or score<best[0]:best=(score,a,b)
    if best is None:raise RuntimeError('Insufficient synthetic groups for split')
    return best[1],best[2]
split_path=COMMON['splits']/'synthetic_frozen_v1.csv'
if SPLIT_MODE=='reuse' and not E3_FROZEN_SPLIT_HINT:
    raise FileNotFoundError('Attach canonical synthetic split for reuse mode')
if E3_FROZEN_SPLIT_HINT is not None:
    frozen_input=Path(E3_FROZEN_SPLIT_HINT)
    assert frozen_input.is_file(),f'Attach frozen E3 split: {frozen_input}'
    if frozen_input.suffix.lower()=='.zip':
        with zipfile.ZipFile(frozen_input) as z:
            key='splits/synthetic_frozen_v1.csv'
            assert key in z.namelist(),f'No {key} in {frozen_input}'
            split_path.write_bytes(z.read(key))
    else:
        shutil.copy2(frozen_input,split_path)
if PRIOR_E3_OUTPUT:
    prior=Path(PRIOR_E3_OUTPUT)/'splits'/'synthetic_frozen_v1.csv'
    assert prior.is_file(),f'Missing prior synthetic split: {prior}'
    if not split_path.is_file():shutil.copy2(prior,split_path)
# Each account computes the synthetic prompt/group split directly and saves it.
# Across accounts compare saved fingerprints before merging results.
if split_path.exists():
    saved=pd.read_csv(split_path,dtype={'relative_path':str,'sha256':str,'dataset_fingerprint':str})
    assert len(saved)==len(syn) and saved.relative_path.is_unique and set(saved.relative_path)==set(syn.relative_path)
    assert set(saved.dataset_fingerprint)=={syn_fingerprint},'Synthetic dataset fingerprint differs from previous run'
    joined=syn[['relative_path','sha256','class','group']].merge(saved,on='relative_path',suffixes=('_now','_saved'),validate='one_to_one')
    assert (joined.sha256_now==joined.sha256_saved).all() and (joined.class_now==joined.class_saved).all()
    assert (joined.group_now.astype(str)==joined.group_saved.astype(str)).all(), 'Synthetic grouping changed'
    syn['split']=syn.relative_path.map(saved.set_index('relative_path')['split'])
else:
    tr,tmp=best_group_split(syn.label,syn.group,.30,SEED)
    va,te=best_group_split(syn.label.iloc[tmp],syn.group.iloc[tmp],.5,SEED+3000)
    assign=np.empty(len(syn),dtype=object)
    assign[tr]='train';assign[tmp[va]]='val';assign[tmp[te]]='test'
    syn['split']=assign
assert syn.groupby('group').split.nunique().max()==1,'Synthetic visual group overlap'
assert syn.loc[syn.prompt_id!=''].groupby(['class','prompt_id']).split.nunique().max()==1, 'Synthetic variants of the same prompt crossed partitions'
assert syn.groupby('sha256').split.nunique().max()==1,'Synthetic SHA256 split overlap'
assert all(set(syn.loc[syn.split==s,'class'])==set(CLASSES) for s in ['train','val','test']), 'Synthetic split missing class'
syn['dataset_fingerprint']=syn_fingerprint
syn[['relative_path','sha256','class','label','group','split','dataset_fingerprint','prompt_id','gan_subtype']].to_csv(split_path,index=False)
syn['source']='synthetic';syn['record_id']='synthetic:'+syn.relative_path
syn['group_key']='synthetic:'+syn.group.astype(str)
logger.info('Frozen synthetic split: %s',syn.split.value_counts().to_dict())

## 5. Full-pool MD5 audit, frozen holdouts and independent similarity controls

Join complete train/validation/test partitions **before** selecting any mixture. Check MD5/SHA-256 and group overlap, save a one-row-per-image final disposition table and a source-specific summary in `audits/md5_audit_full_pools.csv`. This report distinguishes MD5 duplicates, conflicting labels and visual/prompt quarantine; the old 270-removal count is NOT copied to the new dataset. Save holdout fingerprints and independent EfficientNet similarity rates at both cosine thresholds 0.90 and 0.95.

In [ ]:
def pool(df,split):return df.loc[df.split==split].copy().reset_index(drop=True)
real_train,real_val,real_test=[pool(real,x) for x in ['train','val','test']]
syn_train,syn_val,syn_test=[pool(syn,x) for x in ['train','val','test']]
full_train=pd.concat([real_train,syn_train],ignore_index=True)
fixed_val=pd.concat([real_val,syn_val],ignore_index=True)
fixed_test=pd.concat([real_test,syn_test],ignore_index=True)
all_fixed=pd.concat([full_train.assign(partition='train'),fixed_val.assign(partition='val'),
                     fixed_test.assign(partition='test')],ignore_index=True)
assert all_fixed.record_id.is_unique
assert all_fixed.groupby('sha256').partition.nunique().max()==1,'Residual exact-hash leakage; resolve SYNTHETIC side before training'
assert all_fixed.groupby('group_key').partition.nunique().max()==1,'Group leakage'
assert not (set(full_train.sha256)&set(fixed_val.sha256)), 'Train–validation hash leakage'
assert not (set(full_train.sha256)&set(fixed_test.sha256)), 'Train–test hash leakage'
assert not (set(fixed_val.sha256)&set(fixed_test.sha256)), 'Validation–test hash leakage'
assert set(real_test.relative_path)==set(real.loc[real.split=='test','relative_path'])
assert set(real_val.relative_path)==set(real.loc[real.split=='val','relative_path'])
assert set(fixed_test['source'])=={'synthetic','real'}
assert all(set(d['class'])==set(CLASSES) for d in [full_train,fixed_val,fixed_test])
all_fixed[['source','relative_path','class','label','sha256','group_key','partition']].to_csv(COMMON['splits']/'combined_frozen_manifest.csv',index=False)
pd.crosstab([all_fixed['source'],all_fixed['class']],all_fixed['partition']).to_csv(COMMON['splits']/'frozen_class_source_counts.csv')
fp=lambda df:hashlib.sha256('\n'.join((df.source+':'+df.relative_path+'|'+df.sha256+'|'+df['class']).sort_values()).encode()).hexdigest()
val_fp,test_fp=fp(fixed_val),fp(fixed_test)
# Historical-style ungrouped splitting diagnostic: recompute leakage on THIS corrected dataset,
# rather than copying the paper's earlier 21.4% figure into a new experiment.
naive_train,naive_tmp=train_test_split(real,test_size=.30,random_state=SEED,stratify=real['label'])
naive_val,naive_test=train_test_split(naive_tmp,test_size=.50,random_state=SEED,stratify=naive_tmp['label'])
train_groups=set(naive_train['group'])
naive_audit=pd.DataFrame([
    {'split':'validation','heldout_n':len(naive_val),'shared_group_with_train_n':int(naive_val['group'].isin(train_groups).sum()),
     'share':float(naive_val['group'].isin(train_groups).mean())},
    {'split':'test','heldout_n':len(naive_test),'shared_group_with_train_n':int(naive_test['group'].isin(train_groups).sum()),
     'share':float(naive_test['group'].isin(train_groups).mean())},
    {'split':'self_computed_real_test','heldout_n':len(real_test),'shared_group_with_train_n':int(real_test['group'].isin(set(real_train['group'])).sum()),'share':0.0}])
naive_audit.to_csv(COMMON['audits']/'naive_split_leakage_control.csv',index=False)
save_json({'real_image_sha256_verified_during_inventory':True,'prior_e2_split_equivalence_verified':False, 'cross_account_split_reuse_mode':SPLIT_MODE=='reuse','real_test_records':len(real_test),'real_val_records':len(real_val),
    'synthetic_test_records':len(syn_test),'synthetic_val_records':len(syn_val),
    'combined_val_fingerprint':val_fp,'combined_test_fingerprint':test_fp,
    'e2_real_fingerprint':str(real.dataset_fingerprint.iloc[0]),'synthetic_fingerprint':syn_fingerprint,
    'combined_sha_overlap':0,'group_overlap':0,'holdouts_changed_after_mixing':False},
    COMMON['splits']/'frozen_holdout_verification.json')
# Source-specific train/test identity for independent residual audit.
if RUN_EFFICIENTNET_AUDIT:
    df_embed=pd.concat([full_train,fixed_val,fixed_test],ignore_index=True)
    E,Ef=embed_paths(df_embed.path.tolist(),backbone='efficientnet')
    train_ix=np.where(np.arange(len(df_embed))<len(full_train))[0]
    ntr=len(full_train);nv=len(fixed_val)
    eval_ids=np.arange(ntr,ntr+nv+len(fixed_test))
    audit=[];suspected=[]
    for part,name in [(eval_ids[:nv],'val'),(eval_ids[nv:],'test')]:
        for eval_src in ['real','synthetic']:
            qidx=part[df_embed.source.iloc[part].values==eval_src]
            for ref_src in ['real','synthetic']:
                ridx=train_ix[df_embed.source.iloc[train_ix].values==ref_src]
                if not len(qidx) or not len(ridx):continue
                scores=[];neighbors=[]
                for start in range(0,len(qidx),CHUNK):
                    qi=qidx[start:start+CHUNK]
                    sim=np.maximum(E[qi]@E[ridx].T,E[qi]@Ef[ridx].T)
                    top=sim.argmax(axis=1)
                    scores.extend(sim[np.arange(len(qi)),top]);neighbors.extend(ridx[top])
                scores=np.asarray(scores);neighbors=np.asarray(neighbors)
                row={'evaluation_split':name,'query_source':eval_src,'train_source':ref_src,
                     'n_queries':len(qidx),'median_cosine':float(np.median(scores))}
                for threshold in AUDIT_THRESHOLDS:
                    row[f'share_ge_{threshold:.2f}']=float((scores>=threshold).mean())
                    row[f'n_ge_{threshold:.2f}']=int((scores>=threshold).sum())
                audit.append(row)
                for query,ref,score in zip(qidx,neighbors,scores):
                    if score>=min(AUDIT_THRESHOLDS):suspected.append({
                        'evaluation_split':name,'query_source':eval_src,'train_source':ref_src,
                        'query_relative_path':df_embed.relative_path.iloc[query],
                        'neighbor_relative_path':df_embed.relative_path.iloc[ref],
                        'cosine':float(score)})
    pd.DataFrame(audit).to_csv(COMMON['audits']/'efficientnet_source_similarity.csv',index=False)
    pd.DataFrame(suspected,columns=['evaluation_split','query_source','train_source','query_relative_path',
                'neighbor_relative_path','cosine']).to_csv(COMMON['audits']/'efficientnet_flagged_pairs.csv',index=False)
    # Whole-image embedding source classifier (train-fit, fixed test evaluation).
    # Unlike clinical disease accuracy, this quantifies whether source is predictable from image appearance.
    full_train_indices=np.arange(ntr)
    test_indices=np.arange(ntr+nv,len(df_embed))
    src_model=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED))
    src_y=(df_embed.source=='synthetic').astype(int).to_numpy()
    src_model.fit(E[full_train_indices],src_y[full_train_indices])
    src_prob=src_model.predict_proba(E[test_indices])[:,1]
    src_pred=(src_prob>=.5).astype(int)
    pd.DataFrame([{'control':'full_image_efficientnet_embedding','test_n':len(test_indices),
        'accuracy':accuracy_score(src_y[test_indices],src_pred),
        'balanced_accuracy':balanced_accuracy_score(src_y[test_indices],src_pred),
        'macro_f1':f1_score(src_y[test_indices],src_pred,average='macro',zero_division=0),
        'source_roc_auc':roc_auc_score(src_y[test_indices],src_prob)}]).to_csv(
            COMMON['audits']/'source_classifier_full_image.csv',index=False)
    del E,Ef;gc.collect()
else:
    pd.DataFrame(columns=['evaluation_split','query_source','train_source','n_queries',
      'median_cosine','share_ge_0.90','n_ge_0.90','share_ge_0.95','n_ge_0.95']).to_csv(
        COMMON['audits']/'efficientnet_source_similarity.csv',index=False)
    pd.DataFrame(columns=['control','test_n','accuracy','balanced_accuracy','macro_f1','source_roc_auc']).to_csv(
        COMMON['audits']/'source_classifier_full_image.csv',index=False)
    pd.DataFrame(columns=['evaluation_split','query_source','train_source','query_relative_path',
      'neighbor_relative_path','cosine']).to_csv(COMMON['audits']/'efficientnet_flagged_pairs.csv',index=False)
# Full-pool MD5 accounting BEFORE ANY mixture selection.
# One final disposition per image (no double counting across SHA, MD5, visual and prompt screens).
real_dispositions=manifest[['relative_path','class','sha256','md5','status']].copy()
real_dispositions['source']='real'
real_dispositions['final_status']=real_dispositions['status']
syn_dispositions=synth_all[['relative_path','class','sha256','md5','final_status']].copy()
syn_dispositions['source']='synthetic'
full_dispositions=pd.concat([real_dispositions[['source','relative_path','class','sha256','md5','final_status']],
                             syn_dispositions[['source','relative_path','class','sha256','md5','final_status']]],ignore_index=True)
assert full_dispositions.duplicated(['source','relative_path']).sum()==0
full_dispositions.to_csv(COMMON['audits']/'full_pool_md5_dispositions.csv',index=False)
full_dispositions.groupby(['source','final_status']).size().rename('image_count').reset_index().to_csv(
    COMMON['audits']/'duplicate_removals_by_source.csv',index=False)
md5_exclusions={'removed_exact_duplicate','removed_synthetic_exact_copy','removed_shared_exact_real',
                'quarantined_synthetic_label_conflict','quarantined_cross_class_exact'}
summary_rows=[]
for source in ('real','synthetic'):
    dd=full_dispositions.loc[full_dispositions.source==source]
    status=dd.final_status
    image_removed=int((status!='eligible').sum())
    summary_rows.append({'source':source,'decoded_candidates':len(dd),
        'md5_exact_duplicate_removed':int((status.isin({'removed_exact_duplicate','removed_synthetic_exact_copy','removed_shared_exact_real'})).sum()),
        'md5_conflicting_label_excluded':int(status.isin({'quarantined_synthetic_label_conflict','quarantined_cross_class_exact'}).sum()),
        'visual_or_prompt_quarantined':int((~status.isin(md5_exclusions|{'eligible'})).sum()),
        'total_removed_or_quarantined':image_removed,'final_retained':int((status=='eligible').sum())})
    assert len(dd)==image_removed+int((status=='eligible').sum()),'Inconsistent disposition totals'
md5_summary=pd.DataFrame(summary_rows)
md5_summary.to_csv(COMMON['audits']/'md5_audit_full_pools.csv',index=False)
# Final exact MD5 check is run on full unsampled partitions, never on a selected 30:70 pool.
md5_cross_partition=int(all_fixed.groupby('md5').partition.nunique().gt(1).sum())
assert md5_cross_partition==0,'MD5 leakage after full-pool audit'
logger.info('Full-pool removal summary (pre-sampling):\n%s',md5_summary.to_string(index=False))
# Provenance-oriented MD5 summary. No train-fraction-specific deletion is permitted.
save_json({'synthetic_excluded_exact_real':int((synth_all.status=='removed_shared_exact_real').sum()),
           'synthetic_excluded_cross_class_exact':int((synth_all.status=='quarantined_synthetic_label_conflict').sum()),
           'synthetic_removed_same_class_exact':int((synth_all.status=='removed_synthetic_exact_copy').sum()),
           'synthetic_excluded_visual_or_cross_source':int((len(synth_all.loc[synth_all.status=='eligible']))-len(syn)),
           'real_holdout_removed':0,'sha256_cross_partition_remaining':0,
           'md5_cross_partition_remaining':md5_cross_partition, 'md5_full_pool_csv':'audits/md5_audit_full_pools.csv'},
           COMMON['audits']/'exact_content_audit_summary.json')
assert all_fixed.groupby('md5').partition.nunique().max()==1,'MD5 duplicate conflict across partitions'
logger.info('Final fixed val=%d, test=%d. Real test=%d, synthetic test=%d',
            len(fixed_val),len(fixed_test),len(real_test),len(syn_test))

## 6. Real-versus-synthetic source predictability and shortcut controls

Compute full-image (EfficientNet), image-metadata-only and border-only real-versus-synthetic classifiers plus a real-image Acne–Vitiligo shortcut check. This tests whether **real versus synthetic** origin is visually predictable; it cannot recover the original source dataset for each real photo without provenance metadata. Save the results under `audits/`.

In [ ]:
def metadata_features(df,with_border=False):
    rows=[]
    for p in df.path:
        with Image.open(p) as im:
            im=ImageOps.exif_transpose(im).convert('RGB')
            w,h=im.size;a=np.asarray(im.resize((64,64)),dtype=np.float32)
            border=np.concatenate([a[:7].reshape(-1,3),a[-7:].reshape(-1,3),a[:,:7].reshape(-1,3),a[:,-7:].reshape(-1,3)])
            vals=[w,h,w/h,np.log1p(Path(p).stat().st_size),float(a.std()),float((a<5).all(axis=2).mean())]
            if with_border:vals.extend(border.mean(0).tolist()+border.std(0).tolist())
            rows.append(vals)
    return np.asarray(rows,dtype=np.float32)
def border_pixels(df):
    xs=[]
    for p in df.path:
        a=load_rgb(p,64)/255.0
        # Only a 7px frame; central content zeroed.
        a[7:-7,7:-7,:]=0
        xs.append(a[::4,::4,:].flatten())
    return np.asarray(xs,dtype=np.float32)
source_rows=[]
if RUN_SOURCE_CONTROLS:
    ytr=(full_train.source=='synthetic').astype(int).to_numpy()
    yte=(fixed_test.source=='synthetic').astype(int).to_numpy()
    for tag,make in [('metadata_only',lambda d:metadata_features(d,False)),
                     ('metadata_plus_border_stats',lambda d:metadata_features(d,True)),
                     ('border_pixels_only',border_pixels)]:
        Xtr=make(full_train);Xte=make(fixed_test)
        classifier=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED))
        classifier.fit(Xtr,ytr);pr=classifier.predict_proba(Xte)[:,1];pred=(pr>=.5).astype(int)
        source_rows.append({'control':tag,'test_n':len(yte),'accuracy':accuracy_score(yte,pred),
            'balanced_accuracy':balanced_accuracy_score(yte,pred),
            'macro_f1':f1_score(yte,pred,average='macro',zero_division=0),
            'source_roc_auc':roc_auc_score(yte,pr) if len(set(yte))==2 else np.nan})
# Standalone Acne–Vitiligo metadata shortcut control from corrected REAL records.
# This is *not* a medical classifier; it checks whether simple metadata distinguishes two disease labels.
binary_tr=real_train.loc[real_train['class'].isin(['Acne','Vitiligo'])]
binary_te=real_test.loc[real_test['class'].isin(['Acne','Vitiligo'])]
control_av=[]
if RUN_SOURCE_CONTROLS:
    for label,make in [('metadata_only',lambda d:metadata_features(d,False)),
                       ('border_pixels_only',border_pixels)]:
        xtr=make(binary_tr);xte=make(binary_te)
        ytr=(binary_tr['class']=='Vitiligo').astype(int).to_numpy()
        yte=(binary_te['class']=='Vitiligo').astype(int).to_numpy()
        check=make_pipeline(StandardScaler(),LogisticRegression(max_iter=1000,random_state=SEED))
        check.fit(xtr,ytr);yp=check.predict(xte)
        control_av.append({'control':label,'train_n':len(ytr),'test_n':len(yte),
            'accuracy':accuracy_score(yte,yp),'balanced_accuracy':balanced_accuracy_score(yte,yp),
            'macro_f1':f1_score(yte,yp,average='macro',zero_division=0)})
pd.DataFrame(control_av,columns=['control','train_n','test_n','accuracy','balanced_accuracy','macro_f1']).to_csv(
    COMMON['audits']/'acne_vitiligo_shortcut_control.csv',index=False)
image_rows=pd.read_csv(COMMON['audits']/'source_classifier_full_image.csv')
if RUN_SOURCE_CONTROLS and image_rows.empty and RUN_EFFICIENTNET_AUDIT:
    raise RuntimeError('Full-image source-classifier audit was enabled but produced no results.')
source_rows += image_rows.to_dict('records')
pd.DataFrame(source_rows,columns=['control','test_n','accuracy','balanced_accuracy','macro_f1','source_roc_auc']).to_csv(
    COMMON['audits']/'source_classifier_results.csv',index=False)
logger.info('Source-classifier controls: %s',pd.DataFrame(source_rows).to_dict('records'))

## 7 · Shared preprocessing, weighted focal loss and Kaggle preflight

Real and synthetic data both use EXIF-corrected RGB, **aspect-ratio-preserving letterboxing** to 224×224, raw float pixel range `[0,255]`, and architecture-specific normalization inside the model. Augmentation applies **only** to training. Focal loss returns per-example values; Keras receives sample weights from the selected mixture's class frequencies. The preflight decodes actual train/validation batches and performs a tiny weighted Keras gradient update before large training begins.

In [ ]:
def tf_decode(path,label):
    def parse(p):return load_rgb(p.decode('utf-8'),IMG_SIZE)
    x=tf.numpy_function(parse,[path],tf.float32);x.set_shape((IMG_SIZE,IMG_SIZE,3))
    return x,tf.cast(label,tf.int32)
augment_layer=keras.Sequential([layers.RandomFlip('horizontal'),layers.RandomRotation(.04),
    layers.RandomZoom(.04),layers.RandomContrast(.08)],name='train_only_augment')
def create_dataset(df,training=False,weights=None):
    ds=tf.data.Dataset.from_tensor_slices((df.path.astype(str).tolist(),df.label.astype(np.int32).to_numpy()))
    if training:ds=ds.shuffle(len(df),seed=SEED,reshuffle_each_iteration=True)
    ds=ds.map(tf_decode,num_parallel_calls=4)
    if training:
        w=tf.constant(weights,dtype=tf.float32)
        def with_aug(x,y):
            x=augment_layer(x,training=True)
            return tf.clip_by_value(x,0.,255.),y,tf.gather(w,y)
        ds=ds.map(with_aug,num_parallel_calls=4)
    return ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
@keras.utils.register_keras_serializable(package='DermaE3')
class FocalLossPerExample(keras.losses.Loss):
    def __init__(self,gamma=FOCAL_GAMMA,alpha=FOCAL_ALPHA,name='focal_per_example',**kwargs):
        super().__init__(name=name,**kwargs);self.gamma=float(gamma);self.alpha=float(alpha)
    def call(self,y_true,y_pred):
        label=tf.reshape(tf.cast(y_true,tf.int32),[-1]);prob=tf.clip_by_value(tf.cast(y_pred,tf.float32),1e-7,1-1e-7)
        p=tf.gather(prob,label,axis=1,batch_dims=1)
        return -self.alpha*tf.pow(1-p,self.gamma)*tf.math.log(p)
    def get_config(self):return {**super().get_config(),'gamma':self.gamma,'alpha':self.alpha}
_dummy=FocalLossPerExample().call(tf.constant([0,1]),tf.constant([[.9,.1],[.3,.7]]))
assert tuple(_dummy.shape)==(2,), 'Focal loss must be per-example, not scalar'
# This preflight is lightweight but actually trains a model on decoded dataset examples.
_preview=pd.concat([full_train.groupby('class').head(2),fixed_val.groupby('class').head(2)]).drop_duplicates('record_id')
_ds_preview=create_dataset(_preview)
_preview_x,_preview_y=next(iter(_ds_preview))
assert _preview_x.shape[1:]==(IMG_SIZE,IMG_SIZE,3) and np.isfinite(_preview_x.numpy()).all()
_small=keras.Sequential([keras.Input(shape=(IMG_SIZE,IMG_SIZE,3)),layers.Rescaling(1/255.),
    layers.GlobalAveragePooling2D(),layers.Dense(len(CLASSES),activation='softmax')])
_small.compile(optimizer=keras.optimizers.Adam(1e-3),loss=FocalLossPerExample())
preflight_loss=float(_small.train_on_batch(_preview_x[:2],_preview_y[:2],sample_weight=np.array([1.,2.],dtype=np.float32)))
assert np.isfinite(preflight_loss)
save_json({'decoded_batch':int(_preview_x.shape[0]),'img_shape':list(_preview_x.shape[1:]),
           'weighted_tiny_train_loss':preflight_loss,'status':'PASS'},COMMON['config']/'preflight.json')
del _small,_preview_x,_preview_y;gc.collect();keras.backend.clear_session()
print('Preflight passed:',preflight_loss)

## 8 · Four ImageNet backbones, three-phase training, and checkpoint selection

Model architectures remain **EfficientNetV2-B0**, **ResNet-50**, **DenseNet-121**, and **MobileNetV3-Large**. Phase 1 trains the classification head; Phases 2 and 3 progressively unfreeze the final backbone layers (not necessarily the entire backbone). BatchNorm running statistics remain frozen. Model selection and ensemble weights use **validation macro-F1 only**; held-out test data never determines checkpoints or mixture ratios.

Per model and mixture: save three phase histories, a phase summary, all best checkpoints, selected checkpoint metadata, and a final `.keras` copy. The only learning-rate reduction policy is `ReduceLROnPlateau`; it does not conflict with a cosine scheduler.

In [ ]:
@keras.utils.register_keras_serializable(package='DermaE3')
class ResNetPreprocess(layers.Layer):
    def call(self,x):return keras.applications.resnet50.preprocess_input(tf.cast(x,tf.float32))
@keras.utils.register_keras_serializable(package='DermaE3')
class DenseNetPreprocess(layers.Layer):
    def call(self,x):return keras.applications.densenet.preprocess_input(tf.cast(x,tf.float32))
CUSTOM_OBJECTS={'FocalLossPerExample':FocalLossPerExample,'ResNetPreprocess':ResNetPreprocess,
                'DenseNetPreprocess':DenseNetPreprocess}
MODEL_SPECS={'EfficientNetV2B0':(EfficientNetV2B0,50,100),'ResNet50':(ResNet50,50,100),
             'DenseNet121':(DenseNet121,60,120),'MobileNetV3Large':(MobileNetV3Large,25,60)}
def make_backbone(name):
    constructor=MODEL_SPECS[name][0]
    kwargs={'include_top':False,'weights':'imagenet','input_shape':(IMG_SIZE,IMG_SIZE,3)}
    if name in ('EfficientNetV2B0','MobileNetV3Large'):kwargs['include_preprocessing']=True
    return constructor(**kwargs)
def build_classifier(name,backbone):
    inp=keras.Input(shape=(IMG_SIZE,IMG_SIZE,3),name='rgb_0_255');x=inp
    if name=='ResNet50':x=ResNetPreprocess()(x)
    elif name=='DenseNet121':x=DenseNetPreprocess()(x)
    x=backbone(x,training=False)
    x=layers.GlobalAveragePooling2D()(x)
    x=layers.BatchNormalization()(x)
    x=layers.Dense(512,activation='relu',kernel_regularizer=keras.regularizers.l2(L2_REG))(x)
    x=layers.Dropout(DROPOUT)(x)
    x=layers.Dense(256,activation='relu',kernel_regularizer=keras.regularizers.l2(L2_REG))(x)
    x=layers.Dropout(DROPOUT/2)(x)
    out=layers.Dense(len(CLASSES),activation='softmax',dtype='float32')(x)
    return keras.Model(inp,out,name=name)
def set_trainable(base,last_n):
    base.trainable=last_n>0
    if last_n>0:
        for l in base.layers[:-last_n]:l.trainable=False
        for l in base.layers[-last_n:]:l.trainable=not isinstance(l,layers.BatchNormalization)
    return sum(int(x.trainable) for x in base.layers)
class MacroF1Callback(keras.callbacks.Callback):
    def __init__(self,val_ds,y):super().__init__();self.ds=val_ds;self.y=np.asarray(y)
    def on_epoch_end(self,epoch,logs=None):
        logs=logs if logs is not None else {}
        probs=self.model.predict(self.ds,verbose=0)
        logs['val_macro_f1']=float(f1_score(self.y,probs.argmax(axis=1),labels=range(len(CLASSES)),average='macro',zero_division=0))
def train_one(name,dirs,train_ds,val_ds,y_val):
    keras.backend.clear_session();gc.collect()
    backbone=make_backbone(name);model=build_classifier(name,backbone)
    summary=[]
    for phase,n_unfreeze,epochs,lr in [(1,0,EPOCHS[0],LEARNING_RATES[0]),
                                      (2,MODEL_SPECS[name][1],EPOCHS[1],LEARNING_RATES[1]),
                                      (3,MODEL_SPECS[name][2],EPOCHS[2],LEARNING_RATES[2])]:
        n_trainable=set_trainable(backbone,n_unfreeze)
        opt=keras.optimizers.AdamW(learning_rate=lr,weight_decay=1e-5) if phase==3 else keras.optimizers.Adam(learning_rate=lr)
        model.compile(optimizer=opt,loss=FocalLossPerExample(),metrics=[keras.metrics.SparseCategoricalAccuracy(name='accuracy')])
        ckpt=dirs['models']/f'{name}_p{phase}_best.keras'
        callbacks=[MacroF1Callback(val_ds,y_val),
            keras.callbacks.ModelCheckpoint(str(ckpt),monitor='val_macro_f1',mode='max',save_best_only=True),
            keras.callbacks.EarlyStopping(monitor='val_macro_f1',mode='max',patience=5 if phase==1 else 7,restore_best_weights=True),
            keras.callbacks.ReduceLROnPlateau(monitor='val_loss',factor=.5,patience=3,mode='min',min_lr=1e-7),
            keras.callbacks.CSVLogger(str(dirs['training']/f'{name}_p{phase}_epochs.csv'))]
        hist=model.fit(train_ds,validation_data=val_ds,epochs=epochs,callbacks=callbacks,verbose=1)
        assert ckpt.exists(),f'Missing {ckpt}'
        model.load_weights(str(ckpt))
        scores=hist.history['val_macro_f1']
        summary.append({'model':name,'phase':phase,'best_epoch':int(np.argmax(scores))+1,
            'epochs_run':len(scores),'trainable_backbone_layers':n_trainable,
            'best_val_macro_f1':float(max(scores)),'checkpoint':str(ckpt)})
        pd.DataFrame(summary).to_csv(dirs['training']/f'{name}_phase_summary.csv',index=False)
        logger.info('%s phase %d done, best val macro-F1=%.4f',name,phase,max(scores))
    best=max(summary,key=lambda x:x['best_val_macro_f1'])
    final=dirs['models']/f'{name}_final.keras';shutil.copy2(best['checkpoint'],final)
    save_json({'model':name,'best_phase':best['phase'],'validation_macro_f1':best['best_val_macro_f1'],
               'selection':'validation only, max macro-F1','model_path':str(final)},
               dirs['training']/f'{name}_selected_checkpoint.json')
    del model,backbone;gc.collect();keras.backend.clear_session()
    return final
# Save/load a tiny test model to catch preprocessing-layer serialization failures early.
for _name in ('ResNet50','DenseNet121'):
    _backbone=keras.Sequential([keras.Input(shape=(IMG_SIZE,IMG_SIZE,3)),layers.Conv2D(4,3,strides=4)],name='smoke_base')
    _model=build_classifier(_name,_backbone);_p=COMMON['config']/f'_smoke_{_name}.keras'
    _model.save(str(_p))
    _restored=keras.models.load_model(str(_p),custom_objects=CUSTOM_OBJECTS,compile=False)
    assert _restored.predict(np.zeros((1,IMG_SIZE,IMG_SIZE,3),np.float32),verbose=0).shape==(1,len(CLASSES))
    _p.unlink();del _backbone,_model,_restored;gc.collect();keras.backend.clear_session()
logger.info('Four model builders and custom-layer serialization defined')

## 9. Metrics and per-source reporting

For each model and the ensemble, save accuracy, macro/weighted-F1, macro precision/recall, MCC, confusion matrices, ROC curves, per-image probabilities, and **separate real-only and synthetic-only test metrics**. Files `mix_<name>/metrics/per_source_overall.csv` and `per_source_recall.csv` are required for every completed mixture.

In [ ]:
def score_predictions(df,prob,system,mixture,split,dirs,save_plot=True):
    y=df.label.to_numpy().astype(int);p=np.asarray(prob,dtype=np.float32)
    assert p.shape==(len(df),len(CLASSES)) and np.isfinite(p).all(),f'Bad predictions: {system}/{split}'
    yp=p.argmax(axis=1)
    pre,rec,f1,support=precision_recall_fscore_support(y,yp,labels=range(len(CLASSES)),zero_division=0)
    row={'mixture':mixture,'Model':system,'split':split,'n':len(df),'Accuracy':accuracy_score(y,yp),
        'Balanced Accuracy':balanced_accuracy_score(y,yp),
        'Precision (Macro)':float(pre.mean()),'Recall (Macro)':float(rec.mean()),
        'F1 (Macro)':f1_score(y,yp,average='macro',labels=range(len(CLASSES)),zero_division=0),
        'F1 (Weighted)':f1_score(y,yp,average='weighted',zero_division=0),
        'Cohen Kappa':cohen_kappa_score(y,yp),'MCC':matthews_corrcoef(y,yp)}
    try:row['AUC-ROC']=float(roc_auc_score(np.eye(len(CLASSES))[y],p,average='macro'))
    except ValueError:row['AUC-ROC']=np.nan
    save=pd.DataFrame({'source':df.source.to_numpy(),'relative_path':df.relative_path.to_numpy(),
        'sha256':df.sha256.to_numpy(),'true_class':df['class'].to_numpy(),
        'pred_class':np.asarray(CLASSES)[yp],'correct':(yp==y).astype(int)})
    for i,cls in enumerate(CLASSES):save[f'prob_{cls}']=p[:,i]
    save.to_csv(dirs['metrics']/f'{system}_{split}_predictions.csv',index=False)
    pd.DataFrame({'class':CLASSES,'precision':pre,'recall':rec,'f1':f1,'support':support}).to_csv(
        dirs['metrics']/f'{system}_{split}_per_class.csv',index=False)
    cm=confusion_matrix(y,yp,labels=range(len(CLASSES)))
    pd.DataFrame(cm,index=CLASSES,columns=CLASSES).to_csv(dirs['metrics']/f'{system}_{split}_confusion.csv')
    if save_plot and split=='test':
        fig,axes=plt.subplots(1,2,figsize=(13,5))
        sns.heatmap(cm,annot=True,fmt='d',xticklabels=CLASSES,yticklabels=CLASSES,ax=axes[0])
        sns.heatmap(cm/np.maximum(cm.sum(1,keepdims=True),1),annot=True,fmt='.2f',vmin=0,vmax=1,
                    xticklabels=CLASSES,yticklabels=CLASSES,ax=axes[1])
        for ax in axes:ax.set_xlabel('Predicted');ax.set_ylabel('True');ax.tick_params(axis='x',rotation=45)
        fig.suptitle(f'{mixture} · {system} · fixed combined test')
        fig.tight_layout();fig.savefig(dirs['figures']/f'{system}_test_confusion.png',dpi=200,bbox_inches='tight');plt.close(fig)
        fig,ax=plt.subplots(figsize=(7,6));oh=np.eye(len(CLASSES))[y]
        for j,cls in enumerate(CLASSES):
            if len(np.unique(oh[:,j]))==2:
                fpr,tpr,_=roc_curve(oh[:,j],p[:,j]);ax.plot(fpr,tpr,label=cls)
        ax.plot([0,1],[0,1],'k--',linewidth=1);ax.legend(fontsize=8)
        ax.set(xlabel='False positive rate',ylabel='True positive rate',title=f'{mixture}: {system} ROC')
        fig.tight_layout();fig.savefig(dirs['figures']/f'{system}_test_roc.png',dpi=200);plt.close(fig)
    return row

def per_source_metrics(df,prob,system,mixture,dirs):
    y=df.label.to_numpy();p=np.asarray(prob);pred=p.argmax(1)
    overall=[];clsrows=[]
    for source in ['real','synthetic']:
        mask=(df.source==source).to_numpy()
        assert mask.any(),f'Missing {source} test samples'
        yt=y[mask];yp=pred[mask]
        pre,rec,f1,support=precision_recall_fscore_support(yt,yp,labels=range(len(CLASSES)),zero_division=0)
        overall.append({'mixture':mixture,'Model':system,'source':source,'n':int(mask.sum()),
            'Accuracy':accuracy_score(yt,yp),'F1 (Macro)':f1_score(yt,yp,labels=range(len(CLASSES)),average='macro',zero_division=0),
            'Precision (Macro)':float(pre.mean()),'Recall (Macro)':float(rec.mean()),
            'F1 (Weighted)':f1_score(yt,yp,average='weighted',zero_division=0),'MCC':matthews_corrcoef(yt,yp)})
        for k,cls in enumerate(CLASSES):clsrows.append({'mixture':mixture,'Model':system,'source':source,
          'class':cls,'precision':pre[k],'recall':rec[k],'f1':f1[k],'support':support[k]})
    return overall,clsrows

## 10. Three controlled training-pool mixtures

Select the source-specific training fractions **only after** the full-pool hash audit and fixed validation/test assignments. Each mix writes `selected_training_manifest.csv`, its actual source/class counts, four checkpoints and epoch histories, per-source evaluation files, and a validation-weighted ensemble. `30_70` samples 30% of available synthetic training images and 70% of available real training images; it does not enforce an exact 30:70 final count. Use a separate account per mix only with shared frozen CSVs.

In [ ]:
RUN_MODELS=list(MODEL_SPECS)
results_by_mix=[]
val_ds=create_dataset(fixed_val);test_ds=create_dataset(fixed_test)
Y_VAL=fixed_val.label.to_numpy();Y_TEST=fixed_test.label.to_numpy()
def training_subset(frac_syn,frac_real):
    pieces=[]
    for source,frame,fraction in [('synthetic',syn_train,frac_syn),('real',real_train,frac_real)]:
        for cls in CLASSES:
            part=frame.loc[frame['class']==cls].sort_values('record_id')
            n=max(1,int(round(len(part)*fraction)))
            subset=part.sample(n=n,replace=False,random_state=SEED+CLASS_TO_ID[cls]+(100 if source=='synthetic' else 0))
            pieces.append(subset)
    result=pd.concat(pieces,ignore_index=True).sort_values('record_id').reset_index(drop=True)
    assert result.record_id.is_unique and set(result.record_id)<=set(full_train.record_id)
    return result
for mixture in MIXES_TO_RUN:
    fs,fr=MIX_FRACTIONS[mixture]
    selected=training_subset(fs,fr)
    mix_root=ROOT/f'mix_{mixture}'
    if PRIOR_E3_OUTPUT and (Path(PRIOR_E3_OUTPUT)/f'mix_{mixture}'/'mixture_config.json').is_file():
        prior_dir=Path(PRIOR_E3_OUTPUT)/f'mix_{mixture}'
        old_meta=json.loads((prior_dir/'mixture_config.json').read_text())
        assert old_meta['fixed_test_sha256']==test_fp and old_meta['fixed_val_sha256']==val_fp, 'Prior run evaluation mismatch'
        assert old_meta.get('selected_train_fingerprint')==fp(selected), 'Prior run training sample differs'
        shutil.copytree(prior_dir,mix_root,dirs_exist_ok=True)
    dirs={k:mix_root/k for k in ['training','models','metrics','ensemble','figures','logs']}
    for d in dirs.values():d.mkdir(parents=True,exist_ok=True)
    # Each mixture has its own persistent log, in addition to the global run log.
    mix_log=dirs['logs']/'run.log'
    mix_handler=logging.FileHandler(mix_log,encoding='utf-8')
    mix_handler.setFormatter(logging.Formatter('%(asctime)s %(levelname)s %(message)s'))
    logger.addHandler(mix_handler)
    selected[['source','relative_path','class','sha256','group_key']].to_csv(mix_root/'selected_training_manifest.csv',index=False)
    n_source=selected.source.value_counts()
    pd.crosstab(selected['class'],selected.source).to_csv(mix_root/'train_class_source_counts.csv')
    counts=selected.label.value_counts().reindex(range(len(CLASSES)),fill_value=0).to_numpy()
    assert (counts>0).all()
    weights=len(selected)/(len(CLASSES)*counts)
    pd.DataFrame({'class':CLASSES,'count':counts,'sample_weight':weights}).to_csv(dirs['training']/'class_weights.csv',index=False)
    real_n,synthetic_n=int(n_source.get('real',0)),int(n_source.get('synthetic',0))
    mix_meta={'mixture':mixture,'synthetic_fraction_of_pool':fs,'real_fraction_of_pool':fr,
        'real_train_n':real_n,'synthetic_train_n':synthetic_n,
        'actual_synthetic_share':synthetic_n/len(selected),'actual_real_share':real_n/len(selected),
        'selected_n':len(selected),'selected_train_fingerprint':fp(selected),'fixed_val_sha256':val_fp,'fixed_test_sha256':test_fp,
        'selection_seed':SEED,'source_mix_is_not_fixed_training_size':True}
    save_json(mix_meta,mix_root/'mixture_config.json')
    logger.info('MIX %s: train synthetic=%s real=%s, actual share=%.1f%% synthetic',
        mixture,synthetic_n,real_n,100*synthetic_n/len(selected))
    if not RUN_TRAINING:
        results_by_mix.append(mix_meta)
        logger.removeHandler(mix_handler);mix_handler.close()
        continue
    train_ds=create_dataset(selected,training=True,weights=weights)
    x,y,w=next(iter(train_ds.take(1)))
    assert x.shape[1:]==(IMG_SIZE,IMG_SIZE,3) and len(y)==len(w)==len(x)
    model_names=[];preds={};metrics=[];source_overall=[];source_recall=[]
    for name in RUN_MODELS:
        final=dirs['models']/f'{name}_final.keras'
        if not final.exists() and PRIOR_E3_OUTPUT:
            previous=Path(PRIOR_E3_OUTPUT)/f'mix_{mixture}'/'models'/f'{name}_final.keras'
            if previous.is_file():shutil.copy2(previous,final);logger.info('Reused attached checkpoint %s',previous)
        if not final.exists():final=train_one(name,dirs,train_ds,val_ds,Y_VAL)
        else:logger.info('Loading existing model %s',final)
        model=keras.models.load_model(str(final),custom_objects=CUSTOM_OBJECTS,compile=False)
        v_prob=model.predict(val_ds,verbose=0);t_prob=model.predict(test_ds,verbose=0)
        val_score=f1_score(Y_VAL,v_prob.argmax(1),labels=range(len(CLASSES)),average='macro',zero_division=0)
        row=score_predictions(fixed_test,t_prob,name,mixture,'test',dirs)
        row['validation_macro_f1']=float(val_score)
        metrics.append(row)
        score_predictions(fixed_val,v_prob,name,mixture,'val',dirs,save_plot=False)
        source_o,source_r=per_source_metrics(fixed_test,t_prob,name,mixture,dirs)
        source_overall.extend(source_o);source_recall.extend(source_r)
        preds[name]={'val':v_prob,'test':t_prob}
        model_names.append(name)
        pd.DataFrame(metrics).to_csv(dirs['metrics']/'model_comparison.csv',index=False)
        pd.DataFrame(source_overall).to_csv(dirs['metrics']/'per_source_overall.csv',index=False)
        pd.DataFrame(source_recall).to_csv(dirs['metrics']/'per_source_recall.csv',index=False)
        del model,v_prob,t_prob;gc.collect();keras.backend.clear_session()
    assert len(model_names)==4,'All four CNNs required for complete Experiment 3'
    f1s=np.array([f1_score(Y_VAL,preds[n]['val'].argmax(1),labels=range(len(CLASSES)),
                    average='macro',zero_division=0) for n in model_names],dtype=float)
    ensemble_w=f1s/f1s.sum() if f1s.sum()>0 else np.full(len(f1s),1/len(f1s))
    pd.DataFrame({'Model':model_names,'validation_macro_f1':f1s,'weight':ensemble_w}).to_csv(
        dirs['ensemble']/'ensemble_weights.csv',index=False)
    ev=sum(float(w)*preds[n]['val'] for w,n in zip(ensemble_w,model_names))
    et=sum(float(w)*preds[n]['test'] for w,n in zip(ensemble_w,model_names))
    score_predictions(fixed_val,ev,'Ensemble_Weighted',mixture,'val',dirs,save_plot=False)
    ens_row=score_predictions(fixed_test,et,'Ensemble_Weighted',mixture,'test',dirs)
    ens_row['validation_macro_f1']=float(f1_score(Y_VAL,ev.argmax(1),labels=range(len(CLASSES)),average='macro',zero_division=0))
    metrics.append(ens_row)
    source_o,source_r=per_source_metrics(fixed_test,et,'Ensemble_Weighted',mixture,dirs)
    source_overall.extend(source_o);source_recall.extend(source_r)
    pd.DataFrame(metrics).to_csv(dirs['metrics']/'model_comparison_with_ensemble.csv',index=False)
    pd.DataFrame(source_overall).to_csv(dirs['metrics']/'per_source_overall.csv',index=False)
    pd.DataFrame(source_recall).to_csv(dirs['metrics']/'per_source_recall.csv',index=False)
    cmp=pd.DataFrame(metrics)
    fig,ax=plt.subplots(figsize=(10,5));cmp.set_index('Model')[['Accuracy','F1 (Macro)','Recall (Macro)']].plot.bar(ax=ax)
    ax.set(ylim=(0,1),ylabel='Score',title=f'E3 {mixture}: fixed test metrics');ax.tick_params(axis='x',rotation=25)
    fig.tight_layout();fig.savefig(dirs['figures']/'model_comparison_dashboard.png',dpi=220);plt.close(fig)
    pd.DataFrame(metrics).to_csv(dirs['metrics']/'final_leaderboard.csv',index=False)
    results_by_mix.append(mix_meta)
    del preds,train_ds;gc.collect();keras.backend.clear_session()
    logger.removeHandler(mix_handler);mix_handler.close()

## 11. Cross-mixture tables and new figures

Aggregate available mixture results only when their holdout fingerprints match. Save comparisons for all systems, real-test and synthetic-test results, class-wise recalls and publication figures. Do not describe the self-computed E3 real test as identical to an existing E2 run unless separately verified.

In [ ]:
all_overall=[];all_recall=[];all_models=[];mixture_info=[]
for mix in MIX_FRACTIONS:
    mix_dir=ROOT/f'mix_{mix}'
    if PRIOR_E3_OUTPUT and not (mix_dir/'metrics'/'per_source_overall.csv').is_file():
        prior=Path(PRIOR_E3_OUTPUT)/f'mix_{mix}'
        if (prior/'metrics'/'per_source_overall.csv').is_file():
            # Only import previously completed runs with identical frozen fingerprints.
            meta=json.loads((prior/'mixture_config.json').read_text())
            assert meta['fixed_val_sha256']==val_fp and meta['fixed_test_sha256']==test_fp, f'Attached prior {mix} uses different holdout data'
            shutil.copytree(prior,mix_dir,dirs_exist_ok=True,ignore=shutil.ignore_patterns('*.keras','*.h5','*.npy','*.npz'))
    overview=mix_dir/'metrics'/'per_source_overall.csv'
    recall=mix_dir/'metrics'/'per_source_recall.csv'
    models=mix_dir/'metrics'/'model_comparison_with_ensemble.csv'
    meta_path=mix_dir/'mixture_config.json'
    if not all(p.is_file() for p in [overview,recall,models,meta_path]):continue
    meta=json.loads(meta_path.read_text())
    assert meta['fixed_test_sha256']==test_fp and meta['fixed_val_sha256']==val_fp, f'Mismatched frozen heldouts in {mix}'
    all_overall.append(pd.read_csv(overview));all_recall.append(pd.read_csv(recall));all_models.append(pd.read_csv(models));mixture_info.append(meta)
pd.DataFrame(mixture_info).to_csv(COMMON['comparison']/'mixture_actual_counts.csv',index=False)
if all_overall:
    overall=pd.concat(all_overall,ignore_index=True);recall=pd.concat(all_recall,ignore_index=True)
    models=pd.concat(all_models,ignore_index=True)
    overall.to_csv(COMMON['comparison']/'per_source_all_mixes.csv',index=False)
    recall.to_csv(COMMON['comparison']/'classwise_recall_across_mixes.csv',index=False)
    models.to_csv(COMMON['comparison']/'all_mixes_comparison.csv',index=False)
    real_comparison=overall[overall.source=='real'].copy()
    # Optional, strictly verified E2 comparison. E2 tested EXACTLY these real test images.
    try:
        baseline=None;summary=None
        if E2_ORIGIN.lower().endswith('.zip'):
            with zipfile.ZipFile(E2_ORIGIN) as archive:
                if 'metrics/model_comparison_with_ensemble.csv' in archive.namelist() and 'experiment_summary.json' in archive.namelist():
                    from io import BytesIO
                    baseline=pd.read_csv(BytesIO(archive.read('metrics/model_comparison_with_ensemble.csv')))
                    summary=json.loads(archive.read('experiment_summary.json'))
        else:
            origin=Path(E2_ORIGIN)
            if (origin/'metrics/model_comparison_with_ensemble.csv').is_file() and (origin/'experiment_summary.json').is_file():
                baseline=pd.read_csv(origin/'metrics/model_comparison_with_ensemble.csv')
                summary=json.loads((origin/'experiment_summary.json').read_text())
        if baseline is not None:
            assert summary.get('dataset_fingerprint')==real.dataset_fingerprint.iloc[0], 'E2 baseline fingerprint changed'
            for _,item in baseline.iterrows():
                if 'Ensemble' not in str(item.get('Model','')):continue
                real_comparison=pd.concat([real_comparison,pd.DataFrame([{
                    'mixture':'E2_REAL_ONLY','Model':item['Model'],'source':'real','n':len(real_test),
                    'Accuracy':item['Accuracy'],'F1 (Macro)':item['F1 (Macro)'],
                    'Precision (Macro)':item.get('Precision (Macro)',np.nan),
                    'Recall (Macro)':item.get('Recall (Macro)',np.nan),
                    'F1 (Weighted)':item.get('F1 (Weighted)',np.nan),'MCC':item.get('MCC',np.nan)}])],ignore_index=True)
    except (ValueError,KeyError,AssertionError,zipfile.BadZipFile) as exc:
        logger.warning('E2 optional baseline not imported: %s',exc)
    real_comparison.to_csv(COMMON['comparison']/'real_test_only_comparison.csv',index=False)
    overall[overall.source=='synthetic'].to_csv(COMMON['comparison']/'synthetic_test_only_comparison.csv',index=False)
    target=overall[(overall.Model=='Ensemble_Weighted')&(overall.source=='real')].copy()
    if not target.empty:
        fig,ax=plt.subplots(figsize=(9,5))
        xx=np.arange(len(target));wid=.35
        ax.bar(xx-wid/2,target['Accuracy'],wid,label='Accuracy')
        ax.bar(xx+wid/2,target['F1 (Macro)'],wid,label='Macro-F1')
        ax.set_xticks(xx,target.mixture);ax.set_ylim(0,1);ax.legend()
        ax.set_title('Identical frozen real test sets across verified E3 mixtures')
        fig.tight_layout();fig.savefig(COMMON['figures']/'e3_combined_summary.png',dpi=220);plt.close(fig)
else:
    logger.warning('No completed mixture metrics yet; audit-only run')
# New manuscript-candidate figures derived from the current experiment, not renamed historical plots.
fig,ax=plt.subplots(figsize=(9,4));pd.crosstab(all_fixed['class'],all_fixed['source']).reindex(CLASSES).plot.bar(ax=ax)
ax.set_title('Corrected combined source counts by class');ax.set_ylabel('Images');ax.tick_params(axis='x',rotation=30)
fig.tight_layout();fig.savefig(COMMON['figures']/'fig5_combined_class_source_distribution.png',dpi=220);plt.close(fig)
fig,ax=plt.subplots(figsize=(9,4));pd.crosstab(syn['class'],syn['split']).reindex(CLASSES).plot.bar(ax=ax)
ax.set_title('Frozen synthetic grouped split');ax.tick_params(axis='x',rotation=30)
fig.tight_layout();fig.savefig(COMMON['figures']/'fig7_synthetic_grouped_splits.png',dpi=220);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4))
q=synth_all.status.value_counts();q.plot.bar(ax=ax);ax.set_title('Synthetic exact-content audit');ax.tick_params(axis='x',rotation=30)
fig.tight_layout();fig.savefig(COMMON['figures']/'fig8_duplicate_audit.png',dpi=220);plt.close(fig)
fig,ax=plt.subplots(figsize=(8,4));shortcut=pd.read_csv(COMMON['audits']/'source_classifier_results.csv')
if not shortcut.empty:ax.bar(shortcut.control,shortcut.accuracy)
ax.set(ylim=(0,1),title='Source predictability controls');ax.tick_params(axis='x',rotation=40)
fig.tight_layout();fig.savefig(COMMON['figures']/'fig9_source_controls.png',dpi=220,bbox_inches='tight');plt.close(fig)
if all_overall:
    for source,fig_name in [('real','fig15_real_test_comparison.png'),('synthetic','fig16_synthetic_test_comparison.png')]:
        sub=overall[(overall.Model=='Ensemble_Weighted') & (overall.source==source)]
        if not sub.empty:
            fig,ax=plt.subplots(figsize=(8,4));ax.bar(sub.mixture,sub['F1 (Macro)'])
            ax.set(ylim=(0,1),title=f'{source.title()} held-out test: ensemble macro-F1')
            fig.tight_layout();fig.savefig(COMMON['figures']/fig_name,dpi=220);plt.close(fig)
    tab=recall[(recall.Model=='Ensemble_Weighted')&(recall.source=='real')]
    if not tab.empty:
        hm=tab.pivot_table(index='class',columns='mixture',values='recall',aggfunc='mean').reindex(CLASSES)
        fig,ax=plt.subplots(figsize=(8,5));sns.heatmap(hm,annot=True,fmt='.2f',vmin=0,vmax=1,ax=ax)
        ax.set_title('Real test: per-class recall across E3 mixtures')
        fig.tight_layout();fig.savefig(COMMON['figures']/'fig17_real_recall_heatmap.png',dpi=220);plt.close(fig)
logger.info('Completed comparisons for %s',[x['mixture'] for x in mixture_info])

## 12. Evidence validation and GitHub-safe archive

Check the expected files for the selected run, produce `artifact_inventory.csv` and `artifact_validation.json`, and write `dermareal_e3_evidence.zip` without model weights or the private visual-review gallery. Review dataset licence and the inventory before publishing; the archive does not by itself certify correct diagnoses, matching E2 results or GPU reproducibility.

In [ ]:
expected_common=['config/run_config.json','config/environment.json','config/preflight.json',
    'dataset/synthetic_manifest.csv','dataset/synthetic_quality_counts.csv','dataset/synthetic_fingerprint.json',
    'splits/splits_v3.csv','splits/synthetic_frozen_v1.csv','splits/combined_frozen_manifest.csv',
    'splits/frozen_holdout_verification.json','audits/duplicate_removals_by_source.csv','audits/md5_audit_full_pools.csv',
    'audits/full_pool_md5_dispositions.csv',
    'audits/synthetic_group_membership.csv','audits/synthetic_group_summary.csv',
    'audits/visual_quarantine_counts_by_class.csv',
    'audits/efficientnet_source_similarity.csv','audits/efficientnet_flagged_pairs.csv',
    'audits/source_classifier_results.csv','audits/acne_vitiligo_shortcut_control.csv',
    'audits/naive_split_leakage_control.csv','figures/fig5_combined_class_source_distribution.png',
    'figures/fig7_synthetic_grouped_splits.png','figures/fig8_duplicate_audit.png',
    'figures/fig9_source_controls.png','logs/run.log']
expected_per_mix=['selected_training_manifest.csv','mixture_config.json','train_class_source_counts.csv',
    'training/class_weights.csv','metrics/model_comparison.csv','metrics/model_comparison_with_ensemble.csv',
    'metrics/per_source_overall.csv','metrics/per_source_recall.csv','metrics/final_leaderboard.csv',
    'ensemble/ensemble_weights.csv','figures/model_comparison_dashboard.png']
validations={}
for mix in MIX_FRACTIONS:
    base=ROOT/f'mix_{mix}'
    if not (base/'mixture_config.json').exists():continue
    missing=[x for x in expected_per_mix if not (base/x).is_file()]
    if RUN_TRAINING and mix in MIXES_TO_RUN:
        missing += [f'models/{name}_final.keras' for name in MODEL_SPECS if not (base/'models'/f'{name}_final.keras').exists()]
    validations[mix]={'missing':missing,'complete':not missing}
missing_common=[x for x in expected_common if not (ROOT/x).is_file()]
save_json({'common_missing':missing_common,'mixture_status':validations,
    'all_selected_mixes_complete':bool(RUN_TRAINING and all(validations.get(m,{}).get('complete',False) for m in MIXES_TO_RUN)),
    'four_models_per_mix':len(MODEL_SPECS),'real_test_fixed_within_E3':True,'e2_real_test_equivalence_verified':False,
    'shared_across_accounts_requires_reuse_or_matching_fingerprints':True,
    'note':'Presence checks do not certify model accuracy, successful reload, clinical labels, or cross-account comparability until all frozen fingerprints are checked'},ROOT/'artifact_validation.json')
rows=[]
for p in sorted(ROOT.rglob('*')):
    if not p.is_file():continue
    rel=p.relative_to(ROOT)
    public=('private_review' not in rel.parts and p.suffix.lower() not in {'.keras','.h5','.npy','.npz','.pt','.pth','.ckpt'})
    rows.append({'relative_path':rel.as_posix(),'bytes':p.stat().st_size,'include_in_public_evidence':public})
pd.DataFrame(rows,columns=['relative_path','bytes','include_in_public_evidence']).to_csv(ROOT/'artifact_inventory.csv',index=False)
zip_path=ROOT.parent/'dermareal_e3_evidence.zip'
with zipfile.ZipFile(zip_path,'w',compression=zipfile.ZIP_DEFLATED,compresslevel=6) as z:
    for p in sorted(ROOT.rglob('*')):
        if p.is_file() and 'private_review' not in p.relative_to(ROOT).parts and p.suffix.lower() not in {'.keras','.h5','.npy','.npz','.pt','.pth','.ckpt'}:
            z.write(p,arcname=p.relative_to(ROOT).as_posix())
with zipfile.ZipFile(zip_path) as z:assert z.testzip() is None
print('E3 evidence:',zip_path,'| artifacts:',len(rows),'| missing common:',missing_common)
print('Mixture statuses:',validations)
print('Frozen split hashes:',val_fp,test_fp)
if missing_common or (RUN_TRAINING and any(not validations.get(x,{}).get('complete',False) for x in MIXES_TO_RUN)):
    raise RuntimeError('Experiment outputs incomplete. Inspect artifact_validation.json; retain partial checkpoints and logs.')